# Belgium Job–Home Network — Master Data Preparation v1.3.1

This notebook consolidates the Belgian job–home data preparation pipeline into one reproducible workflow. It produces two parallel analytical data systems: (A) municipality × year and (B) origin × destination × year.

Design principles: 2025 565-municipality geography is canonical; missing/suppressed OD cells are never silently converted to zero; residence-side and workplace-side quantities remain separate; pre-COVID transport infrastructure is retained as a fixed 2019 baseline; post-treatment variables are labelled rather than automatically used as controls; the final VAR collection plan is separated from historical request-cache metadata; the direct-download archive is indexed for exploration without blindly cross-joining marginal tables; nationality, origin, generation and work-regime margins are standardised for later heterogeneity analysis using final-plan request parameters when the target category was iterated rather than returned as a CSV column.

Default paths match the project at D:\OneDrive - Universiteit Utrecht\31_BL_Network. The notebook does not run DID models; it prepares auditable inputs for municipality DID/event-study/spatial models and edge-level PPML/network models.

## 0. Configuration

In [ ]:

from __future__ import annotations
from pathlib import Path
from collections import defaultdict
import csv, gzip, hashlib, json, math, re, unicodedata, warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

ROOT = Path(r"D:\OneDrive - Universiteit Utrecht\31_BL_Network")
RAW = ROOT / "raw_data"
OUT = ROOT / "02_output" / "master_data_prep_v1_3_1"
SUPPORT = OUT / "support"
QA_DIR = OUT / "qa"
for p in (OUT, SUPPORT, QA_DIR):
    p.mkdir(parents=True, exist_ok=True)

YEAR_MIN, YEAR_MAX = 2015, 2024
YEARS = list(range(YEAR_MIN, YEAR_MAX + 1))
PRE_YEARS = list(range(2015, 2020))
EXPECTED_N = 565
BASELINE_YEAR = 2019
SHOCK_YEAR = 2020

# Core geography and existing source products
BASEMAP = RAW / "Belgium_VAR_Basemap_2025" / "Belgium_VAR_565_2025.geojson"
POP_XLSX = RAW / "controls" / "Bevolking_per_gemeente.xlsx"
POP_PANEL_EXISTING = RAW / "controls" / "population_density_565" / "Belgium_Population_Area_Density_565_2015_2024.csv"

HOUSE_XLSX = RAW / "controls" / "vastgoed_2010_9999.xlsx"
TAX_XLSX = RAW / "controls" / "TF_PSNL_INC_TAX_MUNTY.xlsx"
ADI_XLSX = RAW / "controls" / "TF_SOC_ADI_MUNTY.xlsx"

# Pre-COVID transport / infrastructure baseline from the separate verified transport notebook
TRANSPORT_ROOT = RAW / "controls" / "transport_accessibility_2019"
TRANSPORT_OUTPUT_DIR = TRANSPORT_ROOT / "output"
TRANSPORT_CSV_EXISTING = TRANSPORT_OUTPUT_DIR / "transport_controls_2019_municipality_565.csv"
TRANSPORT_CODEBOOK_EXISTING = TRANSPORT_OUTPUT_DIR / "transport_controls_2019_codebook.csv"
TRANSPORT_METADATA_EXISTING = TRANSPORT_OUTPUT_DIR / "transport_controls_2019_metadata.json"
TRANSPORT_SNAPSHOT = "190101"
TRANSPORT_GEOFABRIK_URL = f"https://download.geofabrik.de/europe/belgium-{TRANSPORT_SNAPSHOT}-free.shp.zip"
TRANSPORT_ZIP = TRANSPORT_ROOT / "raw" / f"belgium-{TRANSPORT_SNAPSHOT}-free.shp.zip"

CONTROL_PANEL_EXISTING = ROOT / "02_output" / "controls_565" / "municipality_controls_565_2015_2024.parquet"
CONTROL_PANEL_EXISTING_CSV = ROOT / "02_output" / "controls_565" / "municipality_controls_565_2015_2024.csv"

OD_DIR = RAW / "VAR_Belgium_2015_2024" / "02_processed"
OD_FILES = {
    "total": OD_DIR / "od_main_2015_2024_age20_64.csv",
    "age": OD_DIR / "od_by_age_2015_2024.csv",
    "sex": OD_DIR / "od_by_sex_2015_2024_age20_64.csv",
    "education": OD_DIR / "od_by_education_2015_2024_age20_64.csv",
    "employment_status": OD_DIR / "od_by_status_2015_2024_age20_64.csv",
    "sector": OD_DIR / "od_by_sector_2015_2024_age20_64.csv",
}

# Full direct VAR archive collected in the previous step
VAR_DIRECT_ROOT = RAW / "VAR_CSV_Direct_v2"
VAR_FINAL_PLAN = VAR_DIRECT_ROOT / "metadata" / "plan.jsonl"
VAR_PLAN_MANIFEST = VAR_DIRECT_ROOT / "metadata" / "plan_manifest.json"
VAR_TASK_STATUS = VAR_DIRECT_ROOT / "reports" / "task_status.json"
VAR_COLLECTION_SUMMARY = VAR_DIRECT_ROOT / "reports" / "summary.json"

# Telework components used in the earlier Belgian DID work
TELEWORK_XLSX = RAW / "telework_exposure" / "Telework_LFS_STATBEL_nl.xlsx"
TELEWORK_BUILDER = ROOT / "02_output" / "telework_exposure_v4"
WFH_RATE_CSV = TELEWORK_BUILDER / "01_nace_year_wfh_rates_2010_2025.csv"
WFH_WORKPLACE_MARGIN_CSV = TELEWORK_BUILDER / "04_var_published_workplace_margins.csv"

# Runtime choices
PREFER_EXISTING_POP_PANEL = False
PREFER_EXISTING_SOCIO_CONTROL_PANEL = False
PREFER_EXISTING_TRANSPORT_CONTROLS = True
REBUILD_TRANSPORT_IF_MISSING = True
BUILD_FULL_VAR_CATALOG = True
BUILD_EXTENDED_HETEROGENEITY = True
HETERO_AGE_SCOPE = "20-64"
HETERO_RESIDENT_STATUS = "Werkend"
HETERO_WORK_REGIME_STATUS = "Loontrekkend"
HETERO_BASELINE_YEAR = 2019
MATERIALIZE_FULL_VAR_VIEWS = False  # can be huge; inventory is built regardless
CREATE_BALANCED_PAIR_SKELETON = False  # 565^2 * 10 = 3,192,250 rows
TREAT_UNPUBLISHED_OD_AS_ZERO = False  # intentionally False: suppression/missingness is not a verified zero
WRITE_MUNICIPALITY_CSV = True
WRITE_OD_CSV_GZ = False
WRITE_GEOJSON_2024 = True

print("Project:", ROOT)
print("Output:", OUT)


## 1. Helpers and the canonical 2025 municipality crosswalk

In [ ]:

def require(condition, message):
    if not bool(condition):
        raise ValueError(message)

def nis5(x):
    if pd.isna(x):
        return np.nan
    s = re.sub(r"\.0$", "", str(x).strip())
    return s.zfill(5) if s.isdigit() else s

def norm(x):
    if pd.isna(x):
        return ""
    return " ".join(str(x).replace("\xa0", " ").strip().split()).casefold()

def ascii_slug(x):
    s = unicodedata.normalize("NFKD", str(x)).encode("ascii", "ignore").decode().lower()
    return re.sub(r"[^a-z0-9]+", "_", s).strip("_")

def safe_div(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    return np.divide(a, b, out=np.full(np.broadcast(a,b).shape, np.nan), where=np.isfinite(a)&np.isfinite(b)&(b!=0))

def write_table(df, path, csv_also=False):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.suffix.lower() == ".parquet":
        df.to_parquet(path, index=False)
        if csv_also:
            df.to_csv(path.with_suffix(".csv"), index=False, encoding="utf-8-sig")
    else:
        df.to_csv(path, index=False, encoding="utf-8-sig")
    return path

def weighted_quantile(values, weights, probs=(.5,.75,.9)):
    v = np.asarray(values, float)
    w = np.asarray(weights, float)
    keep = np.isfinite(v) & np.isfinite(w) & (w > 0)
    if not keep.any():
        return np.full(len(probs), np.nan)
    v, w = v[keep], w[keep]
    order = np.argsort(v, kind="stable")
    v, w = v[order], w[order]
    cw = np.cumsum(w)
    targets = np.asarray(probs) * cw[-1]
    return v[np.minimum(np.searchsorted(cw, targets, side="left"), len(v)-1)]

MERGERS_2019 = {
    "72042": ["72040", "71047"], "72043": ["72025", "72029"],
    "45068": ["45017", "45057"], "44084": ["44001", "44029"],
    "44083": ["44011", "44049"], "12041": ["12030", "12034"],
    "44085": ["44072", "44036", "44080"],
}
RECODE_2019 = {
    "55022": "58001", "56011": "58002", "56085": "58003", "56087": "58004",
    "52063": "55085", "52043": "55086", "55010": "51067", "55039": "51068",
    "55023": "51069", "54007": "57096", "54010": "57097",
}
MERGERS_2025 = {
    "11002": ["11002", "11007"], "23106": ["23023", "23024", "23032"],
    "37021": ["37012", "37018"], "37022": ["37007", "37015"],
    "44086": ["44012", "44048"], "44087": ["44034", "44073"],
    "46029": ["46014", "44045"], "44088": ["44040", "44043"],
    "46030": ["46003", "46013", "11056"], "73110": ["73006", "73032"],
    "73111": ["73009", "73083"], "71071": ["71069", "71057"],
    "71072": ["71022", "73040"], "82039": ["82003", "82005"],
}

FINAL_RECODE = {}
for target, sources in MERGERS_2019.items():
    for source in sources:
        FINAL_RECODE[source] = target
FINAL_RECODE.update(RECODE_2019)
for target, sources in MERGERS_2025.items():
    for source in sources:
        FINAL_RECODE[source] = target

def to_2025_nis(x):
    c = nis5(x)
    if pd.isna(c):
        return np.nan
    seen = set()
    while c in FINAL_RECODE and c not in seen:
        seen.add(c)
        nxt = FINAL_RECODE[c]
        if nxt == c:
            break
        c = nxt
    return c

def merge_unique(base, table, keys=("nis","year"), label="table"):
    if table is None or len(table) == 0:
        return base
    table = table.copy()
    keys = list(keys)
    require(not table.duplicated(keys).any(), f"{label}: duplicate keys {keys}")
    overlap = [c for c in table.columns if c in base.columns and c not in keys]
    if overlap:
        table = table.drop(columns=overlap)
    return base.merge(table, on=keys, how="left", validate="one_to_one")


## 2. Canonical geography, area, pair distance and contiguity

In [ ]:

require(BASEMAP.exists(), f"Missing basemap: {BASEMAP}")
geo = gpd.read_file(BASEMAP).copy()
geo["nis"] = geo["nis"].map(nis5)
require(len(geo) == EXPECTED_N, f"Expected {EXPECTED_N} municipalities; found {len(geo)}")
require(geo["nis"].is_unique, "Duplicate NIS codes in basemap")
require(geo["node_id"].is_unique, "Duplicate VAR node_id in basemap")

geo_area = geo.to_crs(6933)
geo["area_km2"] = geo_area.geometry.area / 1e6
geo["ln_area_km2"] = np.log(geo["area_km2"])
geo["longitude"] = pd.to_numeric(geo["flow_longitude"], errors="coerce")
geo["latitude"] = pd.to_numeric(geo["flow_latitude"], errors="coerce")
require(geo[["longitude","latitude"]].notna().all().all(), "Missing flow coordinates")

ids = geo["nis"].to_numpy()
lon = np.radians(geo["longitude"].to_numpy(float))
lat = np.radians(geo["latitude"].to_numpy(float))
a = (
    np.sin((lat[:,None]-lat[None,:])/2)**2
    + np.cos(lat[:,None])*np.cos(lat[None,:])
    * np.sin((lon[:,None]-lon[None,:])/2)**2
)
km = 6371.0088 * 2 * np.arcsin(np.sqrt(np.clip(a,0,1)))
np.fill_diagonal(km, 0.0)

pair = pd.DataFrame({
    "home_nis": np.repeat(ids, len(ids)),
    "work_nis": np.tile(ids, len(ids)),
    "distance_km": km.ravel(),
})
attrs = geo.set_index("nis")
pair["same_municipality"] = pair["home_nis"].eq(pair["work_nis"])
pair["same_province"] = pair["home_nis"].map(attrs["prov_nis"]).eq(pair["work_nis"].map(attrs["prov_nis"]))
pair["same_region"] = pair["home_nis"].map(attrs["reg_nis"]).eq(pair["work_nis"].map(attrs["reg_nis"]))

# Queen-style boundary touching; diagonal is not treated as contiguity.
contig_edges = []
try:
    sindex = geo.sindex
    for i, geom in enumerate(geo.geometry):
        for j in sindex.query(geom, predicate="touches"):
            if i < j:
                contig_edges.append((geo.iloc[i]["nis"], geo.iloc[j]["nis"]))
except Exception as exc:
    warnings.warn(f"Spatial index touches failed ({exc}); using slower pairwise touches.")
    geoms = list(geo.geometry)
    for i in range(len(geoms)):
        for j in range(i+1, len(geoms)):
            if geoms[i].touches(geoms[j]):
                contig_edges.append((geo.iloc[i]["nis"], geo.iloc[j]["nis"]))

contig = pd.DataFrame(contig_edges, columns=["nis_i","nis_j"])
pair_key = set((a,b) for a,b in contig_edges) | set((b,a) for a,b in contig_edges)
pair["contiguous"] = [((a,b) in pair_key) for a,b in zip(pair.home_nis, pair.work_nis)]

write_table(pair, SUPPORT / "03_od_pair_static.parquet")
write_table(contig, SUPPORT / "08_spatial_weights" / "contiguity_edges.parquet")
print("Pair rows:", f"{len(pair):,}", "| contiguity undirected edges:", len(contig))


## 3. Population, area and density — existing verified panel or full rebuild

In [ ]:

def expected_mapping_for_year(year, target_codes):
    reverse_2019 = {new: old for old, new in RECODE_2019.items()}
    rows = []
    for target in sorted(target_codes):
        if year < 2025 and target in MERGERS_2025:
            sources, method = MERGERS_2025[target], "sum_pre2025_components"
        elif year < 2019 and target in MERGERS_2019:
            sources, method = MERGERS_2019[target], "sum_pre2019_components"
        elif year < 2019 and target in reverse_2019:
            sources, method = [reverse_2019[target]], "recode_2019"
        else:
            sources, method = [target], "direct"
        for source in sources:
            rows.append({"year":year, "source_nis":source, "nis":target, "mapping_method":method})
    out = pd.DataFrame(rows)
    require(out["source_nis"].is_unique, f"{year}: source municipality maps to multiple targets")
    return out

def _find_population_sheet(xls, year):
    if str(year) in xls.sheet_names:
        return str(year)
    cand = [s for s in xls.sheet_names if re.search(fr"(?<!\d){year}(?!\d)", str(s))]
    require(len(cand)==1, f"Cannot uniquely identify population sheet for {year}: {cand}")
    return cand[0]

def _read_population_year(path, sheet, year):
    preview = pd.read_excel(path, sheet_name=sheet, header=None, nrows=30)
    needed = {"NIS code","Woonplaats","Mannen","Vrouwen","Totaal"}
    header_rows = []
    for i, row in preview.iterrows():
        values = set(row.dropna().astype(str).str.strip())
        if needed.issubset(values):
            header_rows.append(i)
    require(len(header_rows)==1, f"{sheet}: cannot uniquely identify header")
    d = pd.read_excel(path, sheet_name=sheet, header=header_rows[0], dtype=object)
    d.columns = d.columns.astype(str).str.strip()
    d = d[["NIS code","Woonplaats","Mannen","Vrouwen","Totaal"]].rename(columns={
        "NIS code":"source_nis","Woonplaats":"source_name",
        "Mannen":"population_male","Vrouwen":"population_female","Totaal":"population"
    })
    d["source_nis"] = d["source_nis"].map(nis5)
    d = d[d["source_nis"].astype(str).str.fullmatch(r"\d{5}", na=False)].copy()
    for c in ["population","population_male","population_female"]:
        d[c] = pd.to_numeric(d[c], errors="coerce")
    is_aggregate = d["source_nis"].str.endswith("000") | d["source_nis"].isin(["20001","20002"])
    d = d.loc[~is_aggregate].copy()
    d["year"] = year
    return d

def rebuild_population_panel():
    require(POP_XLSX.exists(), f"Missing population workbook: {POP_XLSX}")
    target_codes = set(geo["nis"])
    xls = pd.ExcelFile(POP_XLSX)
    frames = []
    for y in range(YEAR_MIN-1, YEAR_MAX+1):
        sheet = _find_population_sheet(xls, y)
        frames.append(_read_population_year(POP_XLSX, sheet, y))
    rawpop = pd.concat(frames, ignore_index=True)
    cross = pd.concat([expected_mapping_for_year(y,target_codes) for y in range(YEAR_MIN-1,YEAR_MAX+1)], ignore_index=True)

    audits = []
    for y in range(YEAR_MIN-1, YEAR_MAX+1):
        exp = set(cross.loc[cross.year.eq(y),"source_nis"])
        obs = set(rawpop.loc[rawpop.year.eq(y),"source_nis"])
        audits.append({"year":y,"expected_codes":len(exp),"observed_codes":len(obs),"codes_match":exp==obs})
        require(exp==obs, f"{y}: population source codes differ from expected geography")

    mapped = rawpop.merge(cross, on=["year","source_nis"], how="left", validate="one_to_one")
    require(mapped["nis"].notna().all(), "Population has unmapped municipalities")
    pop = mapped.groupby(["nis","year"],as_index=False)[["population","population_male","population_female"]].sum(min_count=1)
    pop = pop.merge(geo[["nis","area_km2","ln_area_km2"]], on="nis", validate="many_to_one")
    pop["population_density_km2"] = pop["population"]/pop["area_km2"]
    pop["ln_population"] = np.log(pop["population"].where(pop["population"]>0))
    pop["ln_population_density"] = np.log(pop["population_density_km2"].where(pop["population_density_km2"]>0))
    pop = pop[pop.year.between(YEAR_MIN,YEAR_MAX)].copy()
    require(len(pop)==EXPECTED_N*len(YEARS), "Population panel is not balanced at 565 x 10")
    pd.DataFrame(audits).to_csv(QA_DIR/"population_rebuild_audit.csv", index=False, encoding="utf-8-sig")
    return pop

if PREFER_EXISTING_POP_PANEL and POP_PANEL_EXISTING.exists():
    population = pd.read_csv(POP_PANEL_EXISTING, dtype={"nis":str})
    population["nis"] = population["nis"].map(nis5)
    population = population[population.year.between(YEAR_MIN,YEAR_MAX)].copy()
    print("Population source: verified existing panel")
else:
    population = rebuild_population_panel()
    print("Population source: rebuilt from raw workbook")

require(len(population)==EXPECTED_N*len(YEARS), f"Population panel rows={len(population)}")
write_table(population, SUPPORT/"population_area_density_565_2015_2024.parquet")


## 4. Housing, fiscal income and administrative disposable income

In [ ]:

HOUSE_TYPE_MAP = {
    "Huizen met 2 of 3 gevels (gesloten + halfopen bebouwing)": "attached_semi",
    "Huizen met 4 of meer gevels (open bebouwing)": "detached",
    "Alle huizen met 2, 3, 4 of meer gevels (excl. appartementen)": "all_houses_excl_apartment",
    "Appartementen": "apartment",
}

def load_house_prices(path):
    frames = []
    xls = pd.ExcelFile(path)
    for year in YEARS:
        if str(year) not in xls.sheet_names:
            warnings.warn(f"Housing sheet {year} not found")
            continue
        d = pd.read_excel(path, sheet_name=str(year))
        d = d[
            (pd.to_numeric(d["CD_niveau_refnis"],errors="coerce")==5)
            & d["CD_PERIOD"].astype(str).str.upper().eq("Y")
        ].copy()
        d["year"] = pd.to_numeric(d["CD_YEAR"], errors="coerce").astype("Int64")
        d["nis"] = d["CD_REFNIS"].map(to_2025_nis)
        d["house_type"] = d["CD_TYPE_NL"].map(HOUSE_TYPE_MAP)
        d = d[d.house_type.notna()].copy()
        frames.append(d[["year","nis","house_type","MS_TOTAL_TRANSACTIONS","MS_P_50_median"]])
    long = pd.concat(frames,ignore_index=True)
    med = long.pivot_table(index=["year","nis"],columns="house_type",values="MS_P_50_median",aggfunc="first")
    med.columns = ["house_price_median_"+c for c in med.columns]
    txn = long.pivot_table(index=["year","nis"],columns="house_type",values="MS_TOTAL_TRANSACTIONS",aggfunc="first")
    txn.columns = ["house_transactions_"+c for c in txn.columns]
    out = med.join(txn,how="outer").reset_index()
    out["housing_cost_main"] = out.get("house_price_median_all_houses_excl_apartment")
    for c in ["housing_cost_main","house_price_median_apartment"]:
        if c in out:
            out["ln_"+c] = np.log(pd.to_numeric(out[c],errors="coerce").where(pd.to_numeric(out[c],errors="coerce")>0))
    return long, out

def load_tax_income(path):
    d = pd.read_excel(path, sheet_name="TF_PSNL_INC_TAX_MUNTY").copy()
    d["year"] = pd.to_numeric(d["CD_YEAR"],errors="coerce").astype("Int64")
    d = d[d.year.between(YEAR_MIN,YEAR_MAX)].copy()
    d["old_nis"] = d["CD_MUNTY_REFNIS"].map(nis5)
    d["nis"] = d["old_nis"].map(to_2025_nis)
    additive = [c for c in [
        "MS_NBR_NON_ZERO_INC","MS_NBR_ZERO_INC","MS_TOT_NET_TAXABLE_INC","MS_TOT_NET_INC",
        "MS_NBR_TOT_NET_INC","MS_REAL_ESTATE_NET_INC","MS_NBR_REAL_ESTATE_NET_INC",
        "MS_TOT_NET_MOV_ASS_INC","MS_NBR_NET_MOV_ASS_INC","MS_TOT_NET_VARIOUS_INC",
        "MS_NBR_NET_VARIOUS_INC","MS_TOT_NET_PROF_INC","MS_NBR_NET_PROF_INC",
        "MS_SEP_TAXABLE_INC","MS_NBR_SEP_TAXABLE_INC","MS_JOINT_TAXABLE_INC",
        "MS_NBR_JOINT_TAXABLE_INC","MS_TOT_DEDUCT_SPEND","MS_NBR_DEDUCT_SPEND",
        "MS_TOT_STATE_TAXES","MS_NBR_STATE_TAXES","MS_TOT_MUNICIP_TAXES",
        "MS_NBR_MUNICIP_TAXES","MS_TOT_SUBURBS_TAXES","MS_NBR_SUBURBS_TAXES",
        "MS_TOT_TAXES","MS_NBR_TOT_TAXES","MS_TOT_RESIDENTS"
    ] if c in d.columns]
    for c in additive:
        d[c] = pd.to_numeric(d[c], errors="coerce")
    out = d.groupby(["year","nis"],as_index=False)[additive].sum(min_count=1)
    if {"MS_TOT_NET_TAXABLE_INC","MS_TOT_RESIDENTS"}.issubset(out):
        out["taxable_income_per_resident"] = out["MS_TOT_NET_TAXABLE_INC"]/out["MS_TOT_RESIDENTS"].replace(0,np.nan)
        out["ln_taxable_income_per_resident"] = np.log(out["taxable_income_per_resident"].where(out["taxable_income_per_resident"]>0))
    if {"MS_TOT_NET_INC","MS_TOT_RESIDENTS"}.issubset(out):
        out["net_income_per_resident"] = out["MS_TOT_NET_INC"]/out["MS_TOT_RESIDENTS"].replace(0,np.nan)
        out["ln_net_income_per_resident"] = np.log(out["net_income_per_resident"].where(out["net_income_per_resident"]>0))
    if {"MS_TOT_TAXES","MS_TOT_RESIDENTS"}.issubset(out):
        out["taxes_per_resident"] = out["MS_TOT_TAXES"]/out["MS_TOT_RESIDENTS"].replace(0,np.nan)
    return d, out

def _wmean(v,w):
    v = pd.to_numeric(v,errors="coerce")
    w = pd.to_numeric(w,errors="coerce")
    ok = v.notna() & w.notna() & (w>0)
    return np.average(v[ok],weights=w[ok]) if ok.any() else np.nan

def load_disposable_income(path):
    d = pd.read_excel(path, sheet_name="TF_SOC_ADI_MUNTY").copy()
    d["year"] = pd.to_numeric(d["CD_YEAR"],errors="coerce").astype("Int64")
    d = d[d.year.between(YEAR_MIN,YEAR_MAX)].copy()
    d["old_nis"] = d["CD_MUNTY_REFNIS"].map(nis5)
    d["nis"] = d["old_nis"].map(to_2025_nis)
    rows = []
    for (year,nis),g in d.groupby(["year","nis"],sort=False):
        eligible = pd.to_numeric(g["MS_NBR_ELIGIBLE"],errors="coerce")
        noneligible = pd.to_numeric(g["MS_NBR_NOT_ELIGIBLE"],errors="coerce")
        es, ns = eligible.sum(min_count=1), noneligible.sum(min_count=1)
        tot = es + ns
        rows.append({
            "year":year,"nis":nis,"adi_n_components":len(g),"adi_eligible_n":es,
            "adi_not_eligible_n":ns,"adi_not_eligible_share":ns/tot if pd.notna(tot) and tot>0 else np.nan,
            "adi_q1_wapprox":_wmean(g["MS_Q1"],eligible),
            "adi_median_wapprox":_wmean(g["MS_MEDIAN"],eligible),
            "adi_q3_wapprox":_wmean(g["MS_Q3"],eligible),
            "adi_iqr_wapprox":_wmean(g["MS_INT_QUART_DIFF"],eligible),
            "adi_arop_share_wapprox":_wmean(g["MS_ADMIN_AROP"],eligible),
            "adi_ioe_hh_share_wapprox":_wmean(g["MS_PERC_IOE_HH"],eligible),
            "adi_quantile_is_approximation":len(g)>1,
        })
    out = pd.DataFrame(rows)
    out["ln_adi_median_wapprox"] = np.log(out["adi_median_wapprox"].where(out["adi_median_wapprox"]>0))
    return d, out

# Existing 136-column control panel is accepted as a verified shortcut for socioeconomic fields.
# To keep the new master interpretable, only housing/income fields are imported here; employment/network
# variables are rebuilt from the OD sources below.
if PREFER_EXISTING_SOCIO_CONTROL_PANEL and (CONTROL_PANEL_EXISTING.exists() or CONTROL_PANEL_EXISTING_CSV.exists()):
    src = CONTROL_PANEL_EXISTING if CONTROL_PANEL_EXISTING.exists() else CONTROL_PANEL_EXISTING_CSV
    old_controls = pd.read_parquet(src) if src.suffix==".parquet" else pd.read_csv(src, dtype={"nis":str})
    old_controls["nis"] = old_controls["nis"].map(nis5)
    keep = ["year","nis"] + [
        c for c in old_controls.columns
        if any(k in c.lower() for k in ["house","housing","taxable_income","net_income","taxes_per",
                                        "adi_","income_per_resident","nonzero_income"])
        and not c.endswith("_2019") and "_premean_" not in c and not c.startswith("lag1_")
    ]
    socio = old_controls[list(dict.fromkeys(keep))].copy()
    print("Socioeconomic controls: imported from verified existing panel", src)
else:
    require(HOUSE_XLSX.exists() and TAX_XLSX.exists() and ADI_XLSX.exists(), "Raw control files missing")
    _, house = load_house_prices(HOUSE_XLSX)
    _, tax = load_tax_income(TAX_XLSX)
    _, adi = load_disposable_income(ADI_XLSX)
    skeleton = pd.MultiIndex.from_product([YEARS,geo.nis],names=["year","nis"]).to_frame(index=False)
    socio = skeleton.merge(house,on=["year","nis"],how="left",validate="one_to_one")
    socio = socio.merge(tax,on=["year","nis"],how="left",validate="one_to_one")
    socio = socio.merge(adi,on=["year","nis"],how="left",validate="one_to_one")
    print("Socioeconomic controls: rebuilt from raw Statbel files")

write_table(socio, SUPPORT/"socioeconomic_controls.parquet")


## 5. Pre-COVID transport accessibility and infrastructure — 2019 OSM baseline

This section integrates the separately verified transport-control workflow into the master preparation pipeline.

**Source and timing.** Infrastructure is measured from the Geofabrik/OpenStreetMap Belgium historical snapshot dated **2019-01-01**, so these are fixed **pre-COVID baseline characteristics**, not annual transport outcomes. The canonical 2025 565-municipality geography is retained.

The section creates two families of variables:

- **Rail / public transport:** stations and halts, local densities, nearest-stop distance, 5/10 km rail-stop counts, gravity rail accessibility, mainline rail length/density, and supplementary all-PT-stop measures.
- **Road:** motorway/trunk/primary/secondary lengths and densities, nearest motorway, nearest motorway-link/ramp, nearest major road, and composite road accessibility.

By default the notebook reuses the already generated 565-municipality CSV. If it is unavailable and `REBUILD_TRANSPORT_IF_MISSING=True`, the raw 2019 Geofabrik archive is downloaded/reused and the indicators are rebuilt.

For causal work these variables are treated as **baseline controls**. Municipality FE absorb their levels; use them for matching/balancing, baseline × year interactions, heterogeneity, or robustness. Broad historical OSM PT-stop counts are supplementary because completeness can vary spatially.

In [ ]:

# ---------------------------------------------------------------------
# 5A. Variable inventory used by the master datasets
# ---------------------------------------------------------------------
TRANSPORT_VARIABLES = [
    "railway_station_count", "railway_halt_count", "rail_stop_count",
    "rail_stop_density_per_100km2", "dist_nearest_rail_stop_km",
    "rail_stop_count_5km", "rail_stop_count_10km", "rail_stop_gravity_10km",
    "pt_stop_count", "pt_stop_density_per_100km2", "dist_nearest_pt_stop_km",
    "pt_stop_count_1km", "pt_stop_count_2km", "pt_stop_count_5km",
    "pt_stop_gravity_2km",
    "rail_km", "light_rail_km", "subway_km", "tram_km",
    "mainline_rail_density_km_per_km2",
    "rail_plus_light_rail_km", "rail_plus_light_rail_density_km_per_km2",
    "motorway_km", "trunk_km", "primary_km", "secondary_km",
    "motorway_link_km", "trunk_link_km", "primary_link_km", "secondary_link_km",
    "major_road_km", "strategic_road_km",
    "motorway_density_km_per_km2", "major_road_density_km_per_km2",
    "strategic_road_density_km_per_km2",
    "dist_nearest_motorway_km", "dist_nearest_motorway_link_km",
    "dist_nearest_major_road_km", "motorway_within_5km", "motorway_within_10km",
    "rail_access_index_z", "road_access_index_z",
]

TRANSPORT_MAIN_CANDIDATES = [
    "rail_stop_gravity_10km",
    "dist_nearest_rail_stop_km",
    "mainline_rail_density_km_per_km2",
    "dist_nearest_motorway_link_km",
    "major_road_density_km_per_km2",
    "rail_access_index_z",
    "road_access_index_z",
]

def _valid_transport_table(d):
    if d is None or len(d) != EXPECTED_N or "nis" not in d.columns:
        return False
    x = d.copy()
    x["nis"] = x["nis"].map(nis5)
    if x["nis"].isna().any() or not x["nis"].is_unique:
        return False
    return set(x["nis"]) == set(geo["nis"])

def load_existing_transport_controls():
    require(TRANSPORT_CSV_EXISTING.exists(),
            f"Transport output not found: {TRANSPORT_CSV_EXISTING}")
    d = pd.read_csv(TRANSPORT_CSV_EXISTING, dtype={"nis": str})
    d["nis"] = d["nis"].map(nis5)
    require(_valid_transport_table(d),
            "Existing transport controls do not match the canonical 565-municipality geography.")
    missing = [c for c in TRANSPORT_MAIN_CANDIDATES if c not in d.columns]
    require(not missing, f"Existing transport output lacks main variables: {missing}")
    return d

# ---------------------------------------------------------------------
# 5B. Full rebuild from the 2019-01-01 Geofabrik / OSM snapshot
# ---------------------------------------------------------------------
def rebuild_transport_controls():
    try:
        import requests
        from tqdm.auto import tqdm
        from scipy.spatial import cKDTree
    except ImportError as exc:
        raise ImportError(
            "Transport rebuild requires requests, tqdm and scipy. "
            "Install them or set PREFER_EXISTING_TRANSPORT_CONTROLS=True."
        ) from exc

    metric_crs = "EPSG:3812"
    raw_dir = TRANSPORT_ROOT / "raw"
    extract_dir = raw_dir / "geofabrik_2019_extracted"
    output_dir = TRANSPORT_OUTPUT_DIR
    for p in [raw_dir, extract_dir, output_dir]:
        p.mkdir(parents=True, exist_ok=True)

    def valid_zip(path):
        if not path.exists() or path.stat().st_size < 1_000_000:
            return False
        import zipfile
        try:
            with zipfile.ZipFile(path, "r") as zf:
                return zf.testzip() is None
        except zipfile.BadZipFile:
            return False

    def download_zip():
        import zipfile
        if valid_zip(TRANSPORT_ZIP):
            print("Using existing transport archive:", TRANSPORT_ZIP)
            return
        TRANSPORT_ZIP.parent.mkdir(parents=True, exist_ok=True)
        tmp = TRANSPORT_ZIP.with_suffix(TRANSPORT_ZIP.suffix + ".part")
        if tmp.exists():
            tmp.unlink()
        headers = {"User-Agent": "Mozilla/5.0 Belgium-master-data-prep/1.1"}
        with requests.get(TRANSPORT_GEOFABRIK_URL, stream=True,
                          timeout=(30, 300), headers=headers) as r:
            r.raise_for_status()
            total = int(r.headers.get("content-length", 0))
            with open(tmp, "wb") as f, tqdm(total=total, unit="B",
                                            unit_scale=True, desc="Transport download") as bar:
                for block in r.iter_content(chunk_size=4*1024*1024):
                    if block:
                        f.write(block)
                        bar.update(len(block))
        tmp.replace(TRANSPORT_ZIP)
        require(valid_zip(TRANSPORT_ZIP), "Downloaded transport ZIP is invalid.")

    def extract_layers():
        import zipfile
        shp = list(extract_dir.rglob("*.shp"))
        tokens = ["transport", "roads", "railways"]
        if shp and all(any(t in p.name.lower() for p in shp) for t in tokens):
            return
        keep_exts = {".shp",".shx",".dbf",".prj",".cpg",".qpj",".qix"}
        with zipfile.ZipFile(TRANSPORT_ZIP, "r") as zf:
            members = []
            for info in zf.infolist():
                name = Path(info.filename).name.lower()
                if Path(name).suffix.lower() in keep_exts and any(
                    t in name for t in ("transport","roads","railways")
                ):
                    members.append(info)
            require(members, "No transport/roads/railways layers in Geofabrik archive.")
            for info in tqdm(members, desc="Extracting transport layers"):
                zf.extract(info, extract_dir)

    def choose_shapefile(kind, area=None):
        out = []
        for p in sorted(extract_dir.rglob("*.shp")):
            n = p.name.lower()
            if kind not in n:
                continue
            is_area = ("_a_" in n) or ("_a." in n)
            if area is True and not is_area:
                continue
            if area is False and is_area:
                continue
            out.append(p)
        return out[0] if out else None

    def normalize_name(x):
        if pd.isna(x):
            return ""
        return re.sub(r"\s+", " ", str(x).strip().lower())

    def pointify_transport(pt_gdf, area_gdf, classes):
        frames = []
        if len(pt_gdf):
            x = pt_gdf[pt_gdf["fclass"].isin(classes)].copy()
            x["source_geom"] = "point"
            frames.append(x)
        if len(area_gdf):
            x = area_gdf[area_gdf["fclass"].isin(classes)].copy()
            if len(x):
                x["geometry"] = x.geometry.representative_point()
                x["source_geom"] = "area"
                frames.append(x)
        if not frames:
            return gpd.GeoDataFrame(columns=["fclass","geometry"], crs=metric_crs)
        keep = ["osm_id","code","fclass","name","geometry","source_geom"]
        out = gpd.GeoDataFrame(
            pd.concat([f[[c for c in keep if c in f.columns]] for f in frames],
                      ignore_index=True),
            crs=metric_crs
        )
        if "name" in out.columns:
            out["_name_norm"] = out["name"].map(normalize_name)
            out["_gx"] = (out.geometry.x/150).round().astype("Int64")
            out["_gy"] = (out.geometry.y/150).round().astype("Int64")
            named = out["_name_norm"].ne("")
            key = out["_name_norm"].fillna("") + "|" + out["_gx"].astype(str) + "|" + out["_gy"].astype(str)
            key.loc[~named] = "unnamed|" + out.index[~named].astype(str)
            out["_dedup_key"] = key
            out = out.drop_duplicates("_dedup_key").copy()
        return out.reset_index(drop=True)

    def point_access_metrics(origins, targets, radii_km, decay_km, cutoff_km, prefix):
        out = origins[["nis"]].copy().reset_index(drop=True)
        if targets.empty:
            out[f"dist_nearest_{prefix}_km"] = np.nan
            for r in radii_km:
                out[f"{prefix}_count_{int(r)}km"] = 0
            out[f"{prefix}_gravity_{decay_km:g}km"] = 0.0
            return out
        o = np.column_stack([origins.geometry.x.to_numpy(), origins.geometry.y.to_numpy()])
        t = np.column_stack([targets.geometry.x.to_numpy(), targets.geometry.y.to_numpy()])
        tree = cKDTree(t)
        nearest_m, _ = tree.query(o, k=1)
        out[f"dist_nearest_{prefix}_km"] = nearest_m / 1000
        for r in radii_km:
            out[f"{prefix}_count_{int(r)}km"] = tree.query_ball_point(
                o, r=r*1000, return_length=True
            ).astype(int)
        gravity = []
        for xy in o:
            inds = tree.query_ball_point(xy, r=cutoff_km*1000)
            if not inds:
                gravity.append(0.0)
                continue
            delta = t[np.asarray(inds)] - xy
            dist_m = np.sqrt((delta**2).sum(axis=1))
            gravity.append(float(np.exp(-dist_m/(decay_km*1000)).sum()))
        out[f"{prefix}_gravity_{decay_km:g}km"] = gravity
        return out

    def counts_inside(points, municipalities, classes, prefix):
        p = points[points["fclass"].isin(classes)].copy()
        if p.empty:
            return pd.DataFrame({"nis": municipalities["nis"], f"{prefix}_count": 0})
        sj = gpd.sjoin(
            p[["fclass","geometry"]],
            municipalities[["nis","geometry"]],
            how="left", predicate="within"
        )
        total = sj.groupby("nis").size().rename(f"{prefix}_count")
        return municipalities[["nis"]].merge(total, on="nis", how="left").fillna(
            {f"{prefix}_count": 0}
        )

    def line_lengths_by_class(lines, municipalities, classes):
        x = lines[lines["fclass"].isin(classes)][["fclass","geometry"]].copy()
        x = x[x.geometry.notna() & ~x.geometry.is_empty]
        if x.empty:
            return municipalities[["nis"]].copy()
        inter = gpd.overlay(
            x, municipalities[["nis","geometry"]],
            how="intersection", keep_geom_type=True
        )
        inter["length_km"] = inter.geometry.length/1000
        tab = inter.pivot_table(
            index="nis", columns="fclass", values="length_km",
            aggfunc="sum", fill_value=0
        )
        tab.columns = [f"{c}_km" for c in tab.columns]
        return tab.reset_index()

    def nearest_line_distance(origins, lines, classes, out_name):
        target = lines[lines["fclass"].isin(classes)][["geometry"]].copy()
        if target.empty:
            return pd.DataFrame({"nis": origins["nis"], out_name: np.nan})
        near = gpd.sjoin_nearest(
            origins[["nis","geometry"]].copy(), target,
            how="left", distance_col="_distance_m"
        )
        d = near.groupby("nis",as_index=False)["_distance_m"].min()
        d[out_name] = d["_distance_m"]/1000
        return d[["nis",out_name]]

    def zscore(s):
        x = pd.to_numeric(s,errors="coerce")
        sd = x.std(ddof=0)
        return (x-x.mean())/sd if np.isfinite(sd) and sd>0 else pd.Series(np.nan,index=s.index)

    download_zip()
    extract_layers()

    transport_pt_path = choose_shapefile("transport", area=False)
    transport_a_path = choose_shapefile("transport", area=True)
    railways_path = choose_shapefile("railways", area=False)
    roads_path = choose_shapefile("roads", area=False)
    require(transport_pt_path and railways_path and roads_path,
            "Required historical OSM transport layers were not found.")

    transport_pt = gpd.read_file(transport_pt_path).to_crs(metric_crs)
    transport_a = (gpd.read_file(transport_a_path).to_crs(metric_crs)
                   if transport_a_path else
                   gpd.GeoDataFrame(columns=transport_pt.columns,crs=metric_crs))
    railways = gpd.read_file(railways_path).to_crs(metric_crs)
    roads = gpd.read_file(roads_path).to_crs(metric_crs)
    for name,gdf in {"transport_pt":transport_pt,"transport_a":transport_a,
                     "railways":railways,"roads":roads}.items():
        require("fclass" in gdf.columns, f"{name}: missing Geofabrik fclass")

    muni_wgs = geo.copy()
    muni = muni_wgs.to_crs(metric_crs).copy()
    muni["area_km2_transport"] = muni.geometry.area/1e6

    lon = pd.to_numeric(muni_wgs["flow_longitude"],errors="coerce")
    lat = pd.to_numeric(muni_wgs["flow_latitude"],errors="coerce")
    valid = lon.between(2,7) & lat.between(49,52)
    if valid.mean() > .95:
        origin_points = gpd.GeoDataFrame(
            muni_wgs[["nis"]].copy(),
            geometry=gpd.points_from_xy(lon,lat,crs="EPSG:4326"),
            crs="EPSG:4326"
        ).to_crs(metric_crs)
    else:
        origin_points = muni[["nis","geometry"]].copy()
        origin_points["geometry"] = origin_points.geometry.representative_point()

    rail_classes = {"railway_station","railway_halt"}
    pt_classes = {"railway_station","railway_halt","tram_stop","bus_stop","bus_station"}
    rail_stops = pointify_transport(transport_pt, transport_a, rail_classes)
    pt_stops = pointify_transport(transport_pt, transport_a, pt_classes)

    rail_access = point_access_metrics(
        origin_points, rail_stops, (5,10), 10, 50, "rail_stop"
    )
    pt_access = point_access_metrics(
        origin_points, pt_stops, (1,2,5), 2, 10, "pt_stop"
    )
    rail_inside = counts_inside(rail_stops, muni, rail_classes, "rail_stop")
    pt_inside = counts_inside(pt_stops, muni, pt_classes, "pt_stop")
    station_inside = counts_inside(rail_stops, muni, {"railway_station"}, "railway_station")
    halt_inside = counts_inside(rail_stops, muni, {"railway_halt"}, "railway_halt")

    rail_controls = muni[["nis","area_km2_transport"]].copy()
    for d in [rail_access,pt_access,rail_inside,pt_inside,station_inside,halt_inside]:
        rail_controls = rail_controls.merge(d,on="nis",how="left")
    for c in ["rail_stop_count","pt_stop_count","railway_station_count","railway_halt_count"]:
        rail_controls[c] = pd.to_numeric(rail_controls[c],errors="coerce").fillna(0)
    rail_controls["rail_stop_density_per_100km2"] = (
        rail_controls["rail_stop_count"]/rail_controls["area_km2_transport"]*100
    )
    rail_controls["pt_stop_density_per_100km2"] = (
        rail_controls["pt_stop_count"]/rail_controls["area_km2_transport"]*100
    )

    rail_len = line_lengths_by_class(
        railways, muni, {"rail","light_rail","subway","tram"}
    )
    rail_controls = rail_controls.merge(rail_len,on="nis",how="left")
    for c in ["rail_km","light_rail_km","subway_km","tram_km"]:
        if c not in rail_controls:
            rail_controls[c] = 0.0
        rail_controls[c] = rail_controls[c].fillna(0.0)
    rail_controls["mainline_rail_density_km_per_km2"] = (
        rail_controls["rail_km"]/rail_controls["area_km2_transport"]
    )
    rail_controls["rail_plus_light_rail_km"] = (
        rail_controls["rail_km"]+rail_controls["light_rail_km"]
    )
    rail_controls["rail_plus_light_rail_density_km_per_km2"] = (
        rail_controls["rail_plus_light_rail_km"]/rail_controls["area_km2_transport"]
    )

    road_classes = {
        "motorway","trunk","primary","secondary",
        "motorway_link","trunk_link","primary_link","secondary_link"
    }
    road_len = line_lengths_by_class(roads,muni,road_classes)
    road_controls = muni[["nis","area_km2_transport"]].copy().merge(
        road_len,on="nis",how="left"
    )
    for c in [
        "motorway_km","trunk_km","primary_km","secondary_km",
        "motorway_link_km","trunk_link_km","primary_link_km","secondary_link_km"
    ]:
        if c not in road_controls:
            road_controls[c] = 0.0
        road_controls[c] = road_controls[c].fillna(0.0)
    road_controls["major_road_km"] = (
        road_controls["motorway_km"]+road_controls["trunk_km"]+road_controls["primary_km"]
    )
    road_controls["strategic_road_km"] = (
        road_controls["major_road_km"]+road_controls["secondary_km"]
    )
    for name,num in [
        ("motorway_density_km_per_km2","motorway_km"),
        ("major_road_density_km_per_km2","major_road_km"),
        ("strategic_road_density_km_per_km2","strategic_road_km"),
    ]:
        road_controls[name] = road_controls[num]/road_controls["area_km2_transport"]

    for d in [
        nearest_line_distance(origin_points,roads,{"motorway"},"dist_nearest_motorway_km"),
        nearest_line_distance(origin_points,roads,{"motorway_link"},"dist_nearest_motorway_link_km"),
        nearest_line_distance(origin_points,roads,{"motorway","trunk","primary"},"dist_nearest_major_road_km"),
    ]:
        road_controls = road_controls.merge(d,on="nis",how="left")
    road_controls["motorway_within_5km"] = (
        road_controls["dist_nearest_motorway_link_km"]<=5
    ).astype("Int64")
    road_controls["motorway_within_10km"] = (
        road_controls["dist_nearest_motorway_link_km"]<=10
    ).astype("Int64")

    controls = muni_wgs[["nis"]].copy()
    controls = controls.merge(
        rail_controls.drop(columns=["area_km2_transport"]),on="nis",how="left"
    )
    controls = controls.merge(
        road_controls.drop(columns=["area_km2_transport"]),on="nis",how="left"
    )
    controls["rail_access_index_z"] = pd.DataFrame({
        "station_density":zscore(controls["rail_stop_density_per_100km2"]),
        "rail_density":zscore(controls["mainline_rail_density_km_per_km2"]),
        "gravity":zscore(controls["rail_stop_gravity_10km"]),
        "inverse_distance":zscore(-controls["dist_nearest_rail_stop_km"]),
    }).mean(axis=1,skipna=True)
    controls["road_access_index_z"] = pd.DataFrame({
        "major_density":zscore(controls["major_road_density_km_per_km2"]),
        "motorway_density":zscore(controls["motorway_density_km_per_km2"]),
        "inverse_motorway_distance":zscore(-controls["dist_nearest_motorway_km"]),
        "inverse_access_distance":zscore(-controls["dist_nearest_motorway_link_km"]),
    }).mean(axis=1,skipna=True)
    controls["transport_snapshot"] = "2019-01-01"
    controls["transport_source"] = "OpenStreetMap / Geofabrik historical Belgium extract"

    controls.to_csv(
        TRANSPORT_CSV_EXISTING,index=False,encoding="utf-8-sig"
    )
    return controls

if PREFER_EXISTING_TRANSPORT_CONTROLS and TRANSPORT_CSV_EXISTING.exists():
    transport_baseline = load_existing_transport_controls()
    transport_build_mode = "existing_verified_output"
elif REBUILD_TRANSPORT_IF_MISSING:
    transport_baseline = rebuild_transport_controls()
    require(_valid_transport_table(transport_baseline),
            "Rebuilt transport table does not match the canonical 565 municipalities.")
    transport_build_mode = "rebuilt_from_2019_osm"
else:
    raise FileNotFoundError(
        f"Transport controls missing: {TRANSPORT_CSV_EXISTING}. "
        "Set REBUILD_TRANSPORT_IF_MISSING=True to rebuild."
    )

# Keep the original transport table as a supporting product.
write_table(transport_baseline, SUPPORT/"transport_controls_2019_baseline.parquet")

# Use the source codebook when present; otherwise provide compact master metadata.
if TRANSPORT_CODEBOOK_EXISTING.exists():
    transport_codebook = pd.read_csv(TRANSPORT_CODEBOOK_EXISTING)
else:
    transport_codebook = pd.DataFrame({
        "variable": TRANSPORT_VARIABLES,
        "category": ["transport"]*len(TRANSPORT_VARIABLES),
        "definition": ["See integrated transport section."]*len(TRANSPORT_VARIABLES),
        "recommended_role": ["baseline"]*len(TRANSPORT_VARIABLES),
    })
transport_codebook["master_variable"] = transport_codebook["variable"].map(
    lambda c: f"{c}_2019" if c in TRANSPORT_VARIABLES else c
)
transport_codebook.to_csv(
    SUPPORT/"transport_controls_2019_codebook_master.csv",
    index=False,encoding="utf-8-sig"
)

# Master datasets receive explicit _2019 suffixes to prevent these fixed
# infrastructure measures being mistaken for annual post-treatment controls.
available_transport = [c for c in TRANSPORT_VARIABLES if c in transport_baseline.columns]
transport_for_merge = transport_baseline[["nis"]+available_transport].copy()
transport_for_merge = transport_for_merge.rename(
    columns={c:f"{c}_2019" for c in available_transport}
)
require(not transport_for_merge.duplicated("nis").any(),"Duplicate NIS in transport baseline")

transport_diag = pd.DataFrame({
    "variable":[f"{c}_2019" for c in available_transport],
    "n_nonmissing":[transport_for_merge[f"{c}_2019"].notna().sum() for c in available_transport],
    "share_nonmissing":[transport_for_merge[f"{c}_2019"].notna().mean() for c in available_transport],
})
transport_diag["build_mode"] = transport_build_mode
transport_diag.to_csv(
    QA_DIR/"transport_controls_2019_coverage.csv",
    index=False,encoding="utf-8-sig"
)

print("Transport mode:", transport_build_mode)
print("Transport municipalities:", transport_for_merge["nis"].nunique())
print("Transport variables merged into masters:", len(available_transport))
print("Preferred rail:", "rail_stop_gravity_10km_2019")
print("Preferred road:", "dist_nearest_motorway_link_km_2019")


## 6. Full VAR final collection plan, historical request cache, and exploration helper

The collector directory contains two conceptually different objects:

1. **Final collection plan** — the explicit `metadata/plan.jsonl` used for the completed optimized run. This is the authoritative set for downstream coverage accounting.
2. **Historical request cache** — every request metadata JSON accumulated during probing, diagnostics, earlier plans and the final run. This is useful for provenance, but must not be mistaken for the final analytical collection plan.

This section validates the final plan hash against `plan_manifest.json`, joins it to `task_status.json`, checks the collector summary, and writes separate final-plan and cache inventories. Downstream view loading uses **only final-plan tasks** by default.

In [ ]:

def _read_json_file(path, default=None):
    path = Path(path)
    if not path.exists():
        return default
    return json.loads(path.read_text(encoding="utf-8"))

def _read_jsonl(path):
    path = Path(path)
    rows = []
    if not path.exists():
        return rows
    with path.open(encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            if not line.strip():
                continue
            try:
                rows.append(json.loads(line))
            except Exception as exc:
                raise ValueError(f"{path}: invalid JSONL at line {line_no}") from exc
    return rows

def _sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def build_request_cache_catalog(root=VAR_DIRECT_ROOT):
    reqdir = Path(root) / "metadata" / "requests"
    rows = []
    if not reqdir.is_dir():
        return pd.DataFrame(), pd.DataFrame()
    for p in reqdir.glob("*.json"):
        try:
            m = json.loads(p.read_text(encoding="utf-8"))
        except Exception:
            continue
        rows.append({
            "request_id": m.get("request_id", p.stem),
            "view": m.get("view"),
            "module": m.get("module"),
            "status": m.get("status"),
            "rows": m.get("rows"),
            "raw_file": m.get("raw_file"),
            "encoding": m.get("encoding"),
            "delimiter": m.get("delimiter"),
            "header_json": json.dumps(m.get("header", []), ensure_ascii=False),
            "requested_parameters_json": json.dumps(
                m.get("requested_parameters", {}), ensure_ascii=False, sort_keys=True
            ),
            "downloaded_utc": m.get("downloaded_utc"),
            "error": m.get("error"),
            "metadata_file": str(p.relative_to(root)),
        })
    cache = pd.DataFrame(rows)
    if cache.empty:
        return cache, pd.DataFrame()
    cache["rows"] = pd.to_numeric(cache["rows"], errors="coerce")
    inv = (
        cache.groupby(["module","view","status"], dropna=False)
        .agg(
            cache_requests=("request_id","count"),
            cache_returned_rows=("rows","sum"),
            cache_nonempty_requests=("rows", lambda s: int((s.fillna(0)>0).sum())),
        )
        .reset_index()
    )
    return cache, inv

def build_final_var_plan_catalog(root=VAR_DIRECT_ROOT):
    root = Path(root)
    plan_path = root / "metadata" / "plan.jsonl"
    manifest_path = root / "metadata" / "plan_manifest.json"
    status_path = root / "reports" / "task_status.json"
    summary_path = root / "reports" / "summary.json"

    require(plan_path.exists(), f"Final VAR plan not found: {plan_path}")
    require(manifest_path.exists(), f"VAR plan manifest not found: {manifest_path}")

    manifest = _read_json_file(manifest_path, {})
    plan_sha = _sha256_file(plan_path)
    if manifest.get("plan_sha256"):
        require(
            plan_sha == manifest["plan_sha256"],
            "Final VAR plan SHA256 does not match plan_manifest.json."
        )

    plan = _read_jsonl(plan_path)
    require(len(plan) > 0, "Final VAR plan is empty.")
    if manifest.get("task_count") is not None:
        require(
            len(plan) == int(manifest["task_count"]),
            f"Final VAR plan has {len(plan)} tasks but manifest reports {manifest['task_count']}."
        )

    statuses = _read_json_file(status_path, {}) or {}
    summary = _read_json_file(summary_path, {}) or {}

    cache, cache_inventory = build_request_cache_catalog(root)
    cache_lookup = (
        cache.drop_duplicates("request_id", keep="last")
        .set_index("request_id").to_dict("index")
        if not cache.empty else {}
    )

    rows = []
    for task in plan:
        rid = str(task.get("request_id", ""))
        view_obj = task.get("view") or {}
        st = statuses.get(rid, {}) or {}
        cm = cache_lookup.get(rid, {}) or {}

        view_name = view_obj.get("name") or st.get("view") or cm.get("view")
        module = view_obj.get("module") or st.get("module") or cm.get("module")
        parameters = task.get("parameters") or {}
        status = st.get("status") or cm.get("status") or "pending"

        rows.append({
            "request_id": rid,
            "module": module,
            "view": view_name,
            "status": status,
            "rows": st.get("rows", cm.get("rows")),
            "raw_file": st.get("raw_file", cm.get("raw_file")),
            "encoding": cm.get("encoding"),
            "delimiter": cm.get("delimiter"),
            "header_json": cm.get("header_json", "[]"),
            "requested_parameters_json": json.dumps(
                parameters, ensure_ascii=False, sort_keys=True
            ),
            "checks_json": json.dumps(
                task.get("checks", []), ensure_ascii=False, sort_keys=True
            ),
            "context_evidence_json": json.dumps(
                task.get("context_evidence", []), ensure_ascii=False, sort_keys=True
            ),
            "plan_mode": task.get("mode"),
            "cache_metadata_found": rid in cache_lookup,
            "is_final_plan_task": True,
        })

    final = pd.DataFrame(rows)
    require(final["request_id"].ne("").all(), "A final VAR plan task has no request_id.")
    require(final["request_id"].is_unique, "Duplicate request_id in final VAR plan.")
    final["rows"] = pd.to_numeric(final["rows"], errors="coerce")

    if summary:
        if summary.get("planned_queries") is not None:
            require(
                int(summary["planned_queries"]) == len(final),
                "Collector summary planned_queries does not equal final plan task count."
            )
        if not summary.get("all_planned_queries_received", False):
            warnings.warn(
                "Collector summary does not certify that all final-plan tasks were received. "
                "The master preparation can continue, but inspect final-plan QA before modelling."
            )

    inventory = (
        final.groupby(["module","view","status"], dropna=False)
        .agg(
            final_plan_requests=("request_id","count"),
            total_returned_rows=("rows","sum"),
            nonempty_requests=("rows", lambda s: int((s.fillna(0)>0).sum())),
            cache_metadata_found=("cache_metadata_found","sum"),
        )
        .reset_index()
    )

    return final, inventory, cache, cache_inventory, manifest, summary

def load_var_direct_view(view_name, max_files=None, parameter_equals=None,
                         use_historical_cache=False):
    catalog = var_request_cache if use_historical_cache else var_catalog
    if catalog is None or catalog.empty:
        raise ValueError("Run the VAR catalogue cell first.")

    q = catalog[(catalog.view==view_name) & (catalog.status=="received")].copy()
    if parameter_equals:
        def ok(txt):
            d = json.loads(txt)
            return all(str(d.get(k)) == str(v) for k,v in parameter_equals.items())
        q = q[q.requested_parameters_json.map(ok)]
    if max_files is not None:
        q = q.head(max_files)

    pieces = []
    for r in q.itertuples(index=False):
        if pd.isna(r.raw_file) or not str(r.raw_file):
            continue
        raw = VAR_DIRECT_ROOT / str(r.raw_file)
        if not raw.exists():
            continue
        params = json.loads(r.requested_parameters_json)
        enc = (r.encoding if hasattr(r,"encoding") and pd.notna(r.encoding) and r.encoding
               else "utf-8-sig")
        sep = (r.delimiter if hasattr(r,"delimiter") and pd.notna(r.delimiter) and r.delimiter
               else ",")
        opener = gzip.open if raw.suffix.lower()==".gz" else open
        with opener(raw, "rt", encoding=enc, errors="replace", newline="") as f:
            d = pd.read_csv(f, sep=sep, low_memory=False)
        d["_request_id"] = r.request_id
        d["_collection_scope"] = (
            "historical_request_cache" if use_historical_cache else "final_collection_plan"
        )
        for k,v in params.items():
            d["req__"+ascii_slug(k)] = v
        pieces.append(d)
    return pd.concat(pieces, ignore_index=True) if pieces else pd.DataFrame()

if BUILD_FULL_VAR_CATALOG:
    (
        var_catalog,
        var_inventory,
        var_request_cache,
        var_request_cache_inventory,
        var_plan_manifest,
        var_collection_summary,
    ) = build_final_var_plan_catalog()

    write_table(var_catalog, SUPPORT/"var_direct_final_plan_catalog.parquet")
    var_inventory.to_csv(
        QA_DIR/"var_direct_final_plan_inventory.csv",
        index=False, encoding="utf-8-sig"
    )

    if not var_request_cache.empty:
        write_table(
            var_request_cache,
            SUPPORT/"var_direct_historical_request_cache_catalog.parquet"
        )
        var_request_cache_inventory.to_csv(
            QA_DIR/"var_direct_historical_request_cache_inventory.csv",
            index=False, encoding="utf-8-sig"
        )

    (QA_DIR/"var_direct_collection_summary.json").write_text(
        json.dumps(var_collection_summary, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    (QA_DIR/"var_direct_plan_manifest.json").write_text(
        json.dumps(var_plan_manifest, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )

    status_counts = var_catalog["status"].value_counts(dropna=False).to_dict()
    print("Final VAR plan tasks:", f"{len(var_catalog):,}")
    print("Final-plan statuses:", status_counts)
    if var_collection_summary:
        print(
            "Collector all_planned_queries_received:",
            var_collection_summary.get("all_planned_queries_received")
        )
        print(
            "full_source_coverage_certified:",
            var_collection_summary.get("full_source_coverage_certified")
        )
    print("Historical request-cache metadata files:", f"{len(var_request_cache):,}")
    print("\nFINAL PLAN inventory:")
    print(var_inventory.sort_values(["module","view","status"]).to_string(index=False))
else:
    var_catalog = pd.DataFrame()
    var_inventory = pd.DataFrame()
    var_request_cache = pd.DataFrame()
    var_request_cache_inventory = pd.DataFrame()
    var_plan_manifest = {}
    var_collection_summary = {}

# Deliberately off by default: materialising all final-plan source responses can be huge.
if MATERIALIZE_FULL_VAR_VIEWS and not var_inventory.empty:
    target = SUPPORT/"var_direct_views"
    target.mkdir(exist_ok=True)
    for view in sorted(
        var_catalog.loc[var_catalog.status.eq("received"),"view"].dropna().unique()
    ):
        if "tabel" not in norm(view) and view not in ("Pendel In - Tabel","Pendel Uit - Tabel"):
            continue
        print("Materialising final-plan view:", view)
        d = load_var_direct_view(view)
        if len(d):
            d.to_parquet(target/(ascii_slug(view)+".parquet"), index=False)


## 7. Load processed OD layers and construct the observed domestic network

In [ ]:

NODE_TO_NIS = geo.set_index("node_id")["nis"].to_dict()
NIS_SET = set(geo["nis"])

def _header(path):
    return pd.read_csv(path,nrows=0,encoding="utf-8-sig").columns.tolist()

def load_od(path, dimension=None, age_scope="20-64"):
    require(Path(path).exists(), f"Missing OD file: {path}")
    cols = _header(path)
    use = ["year","home_id","work_id","workers"]
    dimcol = dimension
    if dimcol and dimcol in cols:
        use.append(dimcol)
    if "age_group" in cols and "age_group" not in use:
        use.append("age_group")
    d = pd.read_csv(path,usecols=use,encoding="utf-8-sig",low_memory=False)
    d["year"] = pd.to_numeric(d["year"],errors="coerce")
    d = d[d.year.isin(YEARS)].copy()
    d["year"] = d["year"].astype(int)
    d["workers"] = pd.to_numeric(d["workers"],errors="coerce")
    require((d["workers"].dropna()>=0).all(), f"Negative OD flow in {path}")
    if dimension != "age" and "age_group" in d.columns and age_scope:
        labels = d["age_group"].astype(str).str.replace("–","-",regex=False)
        mask = labels.str.contains(age_scope,regex=False,na=False)
        if mask.any():
            d = d[mask].copy()
    d["home_id"] = d["home_id"].astype(str).str.strip()
    d["work_id"] = d["work_id"].astype(str).str.strip()
    d["home_nis"] = d["home_id"].map(NODE_TO_NIS)
    d["work_nis"] = d["work_id"].map(NODE_TO_NIS)
    return d

od_main_raw = load_od(OD_FILES["total"])
od_domestic = od_main_raw[od_main_raw.home_nis.notna() & od_main_raw.work_nis.notna()].copy()
od_domestic = od_domestic.groupby(["year","home_nis","work_nis"],as_index=False)["workers"].sum(min_count=1)
od_domestic = od_domestic.merge(pair[["home_nis","work_nis","distance_km","same_municipality",
                                      "same_province","same_region","contiguous"]],
                                on=["home_nis","work_nis"],how="left",validate="many_to_one")
od_domestic["observed_in_source"] = True
od_domestic["pair_id"] = od_domestic["home_nis"]+"__"+od_domestic["work_nis"]

# External-node summaries are retained separately rather than discarded from resident/workplace totals.
external_summary = pd.concat([
    od_main_raw[od_main_raw.home_nis.notna() & od_main_raw.work_nis.isna()]
      .groupby(["year","home_nis"],as_index=False)["workers"].sum(min_count=1)
      .rename(columns={"workers":"resident_workers_external"}).assign(side="residence"),
    od_main_raw[od_main_raw.work_nis.notna() & od_main_raw.home_nis.isna()]
      .groupby(["year","work_nis"],as_index=False)["workers"].sum(min_count=1)
      .rename(columns={"workers":"workplace_workers_external_origin"}).assign(side="workplace"),
],ignore_index=True,sort=False)

write_table(od_domestic, SUPPORT/"od_observed_domestic.parquet")
write_table(external_summary, SUPPORT/"od_external_node_summary.parquet")
print("Observed domestic OD rows:", f"{len(od_domestic):,}")


## 8. Municipality employment scale and residence/workplace composition

In [ ]:

# Basic employment quantities use all mapped resident/workplace records, including one-sided external links.
resident_workers = (
    od_main_raw[od_main_raw.home_nis.notna()]
    .groupby(["year","home_nis"],as_index=False)["workers"].sum(min_count=1)
    .rename(columns={"home_nis":"nis","workers":"resident_workers"})
)
workplace_jobs = (
    od_main_raw[od_main_raw.work_nis.notna()]
    .groupby(["year","work_nis"],as_index=False)["workers"].sum(min_count=1)
    .rename(columns={"work_nis":"nis","workers":"workplace_jobs"})
)
local_workers = (
    od_domestic[od_domestic.same_municipality]
    .groupby(["year","home_nis"],as_index=False)["workers"].sum(min_count=1)
    .rename(columns={"home_nis":"nis","workers":"local_workers"})
)
employment = resident_workers.merge(workplace_jobs,on=["year","nis"],how="outer",validate="one_to_one")
employment = employment.merge(local_workers,on=["year","nis"],how="left",validate="one_to_one")
employment["local_workers"] = employment["local_workers"].fillna(0)
employment = employment.merge(geo[["nis","area_km2"]],on="nis",validate="many_to_one")
employment["employment_density_km2"] = employment["workplace_jobs"]/employment["area_km2"]
employment["jobs_to_resident_workers"] = employment["workplace_jobs"]/employment["resident_workers"].replace(0,np.nan)
employment["resident_local_job_share"] = employment["local_workers"]/employment["resident_workers"].replace(0,np.nan)
employment["workplace_local_worker_share"] = employment["local_workers"]/employment["workplace_jobs"].replace(0,np.nan)
employment["ln_workplace_jobs"] = np.log(employment["workplace_jobs"].where(employment["workplace_jobs"]>0))
employment["ln_resident_workers"] = np.log(employment["resident_workers"].where(employment["resident_workers"]>0))
employment["ln_employment_density_km2"] = np.log(employment["employment_density_km2"].where(employment["employment_density_km2"]>0))
write_table(employment, SUPPORT/"employment_scale_municipality_year.parquet")

DIM_CANON = {
    "education":{
        "hooggeschoold":"high","middengeschoold":"middle","kortgeschoold":"low",
        "onbekend":"unknown","totaal":"total","total":"total"
    },
    "sex":{
        "mannen":"male","man":"male","male":"male","vrouwen":"female","vrouw":"female","female":"female",
        "onbekend":"unknown","mannen en vrouwen":"total","totaal":"total","total":"total"
    },
    "employment_status":{
        "loontrekkend":"employee","loontrekkenden":"employee","employee":"employee",
        "zelfstandig":"self_employed","zelfstandigen":"self_employed",
        "werkend":"working","werkenden":"working","totaal":"total","total":"total"
    }
}

def canonical_group(value, dim):
    s = norm(value)
    return DIM_CANON.get(dim,{}).get(s, ascii_slug(value) if s else "missing")

def dimension_node_shares(path, dimcol, dimname, age_scope="20-64"):
    if not Path(path).exists():
        return pd.DataFrame(), pd.DataFrame()
    d = load_od(path, dimension=dimcol, age_scope=age_scope)
    require(dimcol in d.columns, f"{path}: {dimcol} missing")
    d["category"] = d[dimcol].map(lambda x: canonical_group(x,dimname))
    outputs, longs = [], []
    for side,nodecol,prefix in [("residence","home_nis","resident"),("workplace","work_nis","workplace")]:
        x = d[d[nodecol].notna()].copy()
        g = x.groupby(["year",nodecol,"category"],as_index=False)["workers"].sum(min_count=1)
        g = g.rename(columns={nodecol:"nis"})
        longs.append(g.assign(side=side))
        categories = set(g["category"].dropna().astype(str))
        denom_category = "working" if dimname=="employment_status" and "working" in categories else "total"
        totals = g[g.category.eq(denom_category)][["year","nis","workers"]].rename(columns={"workers":"published_total"})
        excluded_denominators = {denom_category}
        if denom_category != "total":
            excluded_denominators.add("total")
        detail = g[~g.category.isin(excluded_denominators)].copy()
        wide = detail.pivot_table(index=["year","nis"],columns="category",values="workers",aggfunc="sum").reset_index()
        wide.columns = ["year","nis"] + [prefix+"_"+dimname+"_"+c+"_workers" for c in wide.columns[2:]]
        wide = wide.merge(totals,on=["year","nis"],how="left",validate="one_to_one")
        detail_cols = [c for c in wide.columns if c.endswith("_workers")]
        for c in detail_cols:
            wide[c.replace("_workers","_share")] = wide[c]/wide["published_total"].replace(0,np.nan)
        wide = wide.rename(columns={"published_total":prefix+"_"+dimname+"_published_total"})
        outputs.append(wide)
    out = outputs[0].merge(outputs[1],on=["year","nis"],how="outer",validate="one_to_one")
    return pd.concat(longs,ignore_index=True), out

education_long, education_wide = dimension_node_shares(OD_FILES["education"],"education","education")
sex_long, sex_wide = dimension_node_shares(OD_FILES["sex"],"sex","sex")
status_long, status_wide = dimension_node_shares(OD_FILES["employment_status"],"employment_status","employment_status")

for name, table in [("education",education_long),("sex",sex_long),("employment_status",status_long)]:
    if len(table):
        write_table(table,SUPPORT/(name+"_node_long.parquet"))

# Age is preserved in long form because published age intervals overlap.
if OD_FILES["age"].exists():
    age_raw = load_od(OD_FILES["age"],dimension="age_group",age_scope=None)
    age_raw["age_category"] = age_raw["age_group"].astype(str).map(ascii_slug)
    age_parts = []
    for side,nodecol in [("residence","home_nis"),("workplace","work_nis")]:
        q = age_raw[age_raw[nodecol].notna()].groupby(["year",nodecol,"age_category"],as_index=False)["workers"].sum(min_count=1)
        q = q.rename(columns={nodecol:"nis"}).assign(side=side)
        age_parts.append(q)
    age_long = pd.concat(age_parts,ignore_index=True)
    write_table(age_long,SUPPORT/"age_node_long.parquet")

    # Non-overlapping 15-64 composition for compact municipality controls.
    age_core_labels = {"15_24","25_54","55_64"}
    age_core = age_long[age_long.age_category.isin(age_core_labels)].copy()
    age_core_total = age_core.groupby(["side","year","nis"],as_index=False)["workers"].sum(min_count=1).rename(columns={"workers":"age_15_64_known_total"})
    age_core = age_core.merge(age_core_total,on=["side","year","nis"],how="left",validate="many_to_one")
    age_core["share"] = age_core["workers"]/age_core["age_15_64_known_total"].replace(0,np.nan)
    age_wide_parts=[]
    for side,prefix in [("residence","resident"),("workplace","workplace")]:
        q=age_core[age_core.side.eq(side)].pivot_table(index=["year","nis"],columns="age_category",values="share",aggfunc="first").reset_index()
        q.columns=["year","nis"]+[f"{prefix}_age_{c}_share" for c in q.columns[2:]]
        age_wide_parts.append(q)
    age_shares_wide=age_wide_parts[0].merge(age_wide_parts[1],on=["year","nis"],how="outer",validate="one_to_one")
    write_table(age_shares_wide,SUPPORT/"age_15_64_shares_municipality_year.parquet")
else:
    age_long = pd.DataFrame()
    age_shares_wide = pd.DataFrame(columns=["year","nis"])


## 8A. Extended VAR population and labour composition for heterogeneity analysis — final-plan direct parser

The optimized VAR collector stores some demographic dimensions in two different ways: a category can either be **expanded inside the returned CSV** or **iterated as a request parameter** in the final plan. The previous v1.3 parser required the target label to appear as a raw CSV column, which was too restrictive for several BNW residence-side tables.

This revised parser uses both sources of information. It reads the verified final-plan catalogue, recovers target categories from request parameters when necessary, filters nuisance dimensions to the intended scope, and only then aggregates to the harmonised 2025 municipality geography.

No synthetic joint distributions are created. Each demographic margin remains separate. For causal heterogeneity, the notebook creates explicit 2019 baseline shares and keeps contemporaneous annual composition only as descriptive/supporting information.

In [ ]:

# ---------------------------------------------------------------------
# Extended demographic / labour composition from the verified VAR archive
# v1.3.1: target category may come from the raw CSV OR from the request
# parameter in the final plan.
# ---------------------------------------------------------------------

HETERO_SPECS = [
    dict(
        key="resident_nationality",
        side="resident",
        dimension="nationality",
        view="BNW - T5ab (tabel)",
        target_names=("Nationaliteit","Nationaliteitsklasse"),
        preferred_age=HETERO_AGE_SCOPE,
        preferred_status=HETERO_RESIDENT_STATUS,
        expected_min_categories=4,
    ),
    dict(
        key="resident_origin",
        side="resident",
        dimension="origin",
        view="BNW - T6ab (tabel)",
        target_names=("Herkomst","Origine Beperkt","Origine"),
        preferred_age=HETERO_AGE_SCOPE,
        preferred_status=HETERO_RESIDENT_STATUS,
        expected_min_categories=4,
    ),
    dict(
        key="resident_generation",
        side="resident",
        dimension="generation",
        view="BNW - T6cd (tabel)",
        target_names=("Generatie",),
        preferred_age=HETERO_AGE_SCOPE,
        preferred_status=HETERO_RESIDENT_STATUS,
        expected_min_categories=4,
    ),
    dict(
        key="resident_work_regime",
        side="resident",
        dimension="work_regime",
        view="BNW - T4 (tabel)",
        target_names=("Arbeidsregime",),
        preferred_age=HETERO_AGE_SCOPE,
        preferred_status=HETERO_WORK_REGIME_STATUS,
        expected_min_categories=3,
    ),
    dict(
        key="workplace_nationality",
        side="workplace",
        dimension="nationality",
        view="BW - T3a nat (tabel)",
        target_names=("Nationaliteit","Nationaliteitsklasse"),
        preferred_age=HETERO_AGE_SCOPE,
        preferred_status=HETERO_RESIDENT_STATUS,
        expected_min_categories=4,
    ),
    dict(
        key="workplace_origin",
        side="workplace",
        dimension="origin",
        view="BW - T3b herk (tabel)",
        target_names=("Herkomst","Origine Beperkt","Origine"),
        preferred_age=HETERO_AGE_SCOPE,
        preferred_status=HETERO_RESIDENT_STATUS,
        expected_min_categories=4,
    ),
]

EXPECTED_HETERO_FAMILIES=[s["key"] for s in HETERO_SPECS]

def _dash_norm(x):
    return norm(x).replace("–","-").replace("—","-").replace("−","-")

def _is_totalish(x):
    z=_dash_norm(x)
    return z in {
        "","totaal","total","totale","all","alle","alles",
        "mannen en vrouwen","m+v","m/v"
    }

def _param_role(name):
    z=_dash_norm(name)
    if re.search(r"(^|[^a-z])(jaar|year|annee)([^a-z]|$)",z):
        return "year"
    if "leeftijd" in z or re.search(r"(^|_)age($|_)",z):
        return "age"
    if "statuut" in z or "employment" in z or "socio" in z or "arbeidsmarktpositie" in z:
        return "employment_status"
    if "geslacht" in z or "gender" in z or re.search(r"(^|_)sex($|_)",z):
        return "sex"
    if "onderwijs" in z or "opleid" in z or re.search(r"(^|_)edu",z):
        return "education"
    if "nationalit" in z:
        return "nationality"
    if "herkomst" in z or "origine" in z:
        return "origin"
    if "generatie" in z:
        return "generation"
    if "arbeidsregime" in z or "prestatietype" in z or "work_regime" in z:
        return "work_regime"
    if "wse" in z or "sector" in z or "nace" in z:
        return "sector"
    if "geografisch niveau" in z or "geography level" in z:
        return "geography_level"
    if any(t in z for t in ("gemeente","woonplaats","werkplaats","provincie","gewest","referentieregio")):
        return "geography"
    return "other"

def _parse_requested_params(txt):
    try:
        return json.loads(txt) if isinstance(txt,str) and txt else {}
    except Exception:
        return {}

def _matches_name(name,candidates):
    z=_dash_norm(name)
    return any(z==_dash_norm(c) for c in candidates)

def _requested_value(params,candidates):
    exact=[(k,v) for k,v in params.items() if _matches_name(k,candidates)]
    if exact:
        return exact[0][0], exact[0][1]
    # Conservative fallback: allow the candidate token to be embedded in a
    # Tableau URL field name, but do not match unrelated dimensions.
    for k,v in params.items():
        zk=_dash_norm(k)
        if any(_dash_norm(c) in zk for c in candidates):
            return k,v
    return None,None

def _value_matches(value,wanted):
    if wanted is None:
        return True
    a=_dash_norm(value).replace(" ","")
    b=_dash_norm(wanted).replace(" ","")
    return a==b

def _task_year(params):
    for k,v in params.items():
        if _param_role(k)=="year":
            m=re.search(r"(20\d{2})",str(v))
            if m:
                return int(m.group(1))
    return None

def _header_list(header_json):
    try:
        return json.loads(header_json) if isinstance(header_json,str) else list(header_json)
    except Exception:
        return []

def _header_target_col(header_json,candidates):
    headers=_header_list(header_json)
    for c in headers:
        if _matches_name(c,candidates):
            return c
    for c in headers:
        z=_dash_norm(c)
        if any(_dash_norm(x) in z for x in candidates):
            return c
    return None

def _scope_penalty(params,spec):
    """
    Score nuisance request parameters. Lower is better.
    Target-category request parameters are intentionally ignored here.
    """
    score=0
    target_names=spec["target_names"]
    for k,v in params.items():
        if _matches_name(k,target_names):
            continue
        role=_param_role(k)
        zv=_dash_norm(v)

        if role=="year":
            continue
        if role=="geography_level":
            if not ("gemeente" in zv or "municip" in zv):
                score += 500
        elif role=="geography":
            # Optimized collector normally leaves specific municipality selectors
            # empty or removes them from the Cartesian axes.
            if str(v).strip() and not _is_totalish(v):
                score += 250
        elif role=="age":
            if _value_matches(v,spec.get("preferred_age")):
                score += 0
            elif str(v).strip()=="":
                score += 1
            elif _is_totalish(v):
                score += 5
            else:
                score += 100
        elif role=="employment_status":
            if _value_matches(v,spec.get("preferred_status")):
                score += 0
            elif str(v).strip()=="":
                score += 1
            elif _is_totalish(v):
                score += 5
            else:
                score += 100
        elif role in {
            "sex","education","sector","nationality","origin","generation","work_regime"
        }:
            # Every non-target demographic dimension should be at source total
            # or be blank because it was expanded in the returned CSV.
            if str(v).strip()=="" or _is_totalish(v):
                score += 0
            else:
                score += 50
    return score

def select_direct_heterogeneity_tasks(spec):
    """
    Select the minimum set of final-plan requests required to reconstruct one
    marginal distribution.

    If the target category is iterated in the request URL, retain one best task
    per year × requested category. If the target is expanded in the returned
    CSV, retain one best task per year.
    """
    require(BUILD_FULL_VAR_CATALOG and not var_catalog.empty,
            "Run the final VAR catalogue section before heterogeneity extraction.")

    target_view=_dash_norm(spec["view"])
    q=var_catalog[
        var_catalog["view"].map(_dash_norm).eq(target_view)
        & var_catalog["status"].eq("received")
    ].copy()
    if q.empty:
        return q

    q["task_params"]=q["requested_parameters_json"].map(_parse_requested_params)
    q["task_year"]=q["task_params"].map(_task_year)
    q=q[q["task_year"].isna() | q["task_year"].isin(YEARS)].copy()

    request_fields=[]
    request_values=[]
    raw_target_cols=[]
    modes=[]
    penalties=[]

    for row in q.itertuples(index=False):
        params=row.task_params
        field,val=_requested_value(params,spec["target_names"])
        raw_col=_header_target_col(row.header_json,spec["target_names"])

        # Non-empty target request => category is being iterated by collector.
        if field is not None and str(val).strip()!="":
            mode="request_parameter"
        elif raw_col is not None:
            mode="raw_column"
        elif field is not None:
            # Empty target filter without a visible target column cannot identify
            # categories and is therefore not usable for this margin.
            mode="unidentified"
        else:
            mode="unidentified"

        request_fields.append(field)
        request_values.append(val)
        raw_target_cols.append(raw_col)
        modes.append(mode)
        penalties.append(_scope_penalty(params,spec))

    q["target_request_field"]=request_fields
    q["target_request_value"]=request_values
    q["target_raw_column"]=raw_target_cols
    q["target_mode"]=modes
    q["scope_penalty"]=penalties
    q=q[q["target_mode"].ne("unidentified")].copy()
    if q.empty:
        return q

    selected=[]

    # Prefer request-parameter tasks when present because they explicitly encode
    # the category and were generated by the collector's expanded Cartesian plan.
    req=q[q["target_mode"].eq("request_parameter")].copy()
    if len(req):
        req["target_request_norm"]=req["target_request_value"].map(_dash_norm)
        group_cols=["task_year","target_request_norm"]
        req=req.sort_values(
            ["scope_penalty","rows","request_id"],
            ascending=[True,True,True]
        )
        selected.append(req.groupby(group_cols,dropna=False,as_index=False).head(1))

    raw=q[q["target_mode"].eq("raw_column")].copy()
    if len(raw):
        # Only add a raw-expanded task for years not already represented by an
        # explicit request-parameter category set.
        represented=set(req["task_year"].dropna().tolist()) if len(req) else set()
        raw=raw[~raw["task_year"].isin(represented)].copy()
        if len(raw):
            raw=raw.sort_values(
                ["scope_penalty","rows","request_id"],
                ascending=[True,True,True]
            )
            selected.append(raw.groupby("task_year",dropna=False,as_index=False).head(1))

    out=pd.concat(selected,ignore_index=True) if selected else q.head(0)
    return out.sort_values(
        ["task_year","target_mode","target_request_value","scope_penalty","request_id"],
        na_position="first"
    ).reset_index(drop=True)

def _read_var_task(row):
    raw=VAR_DIRECT_ROOT / str(row.raw_file)
    require(raw.exists(),f"Missing VAR raw file: {raw}")
    enc=row.encoding if pd.notna(row.encoding) and str(row.encoding) else "utf-8-sig"
    sep=row.delimiter if pd.notna(row.delimiter) and str(row.delimiter) else ","
    opener=gzip.open if raw.suffix.lower()==".gz" else open
    with opener(raw,"rt",encoding=enc,errors="replace",newline="") as f:
        return pd.read_csv(f,sep=sep,low_memory=False)

def _find_raw_col(columns,candidates=(),contains=()):
    cols=list(columns)
    normalized={c:_dash_norm(c) for c in cols}
    for cand in candidates:
        z=_dash_norm(cand)
        for c,n in normalized.items():
            if n==z:
                return c
    for token in contains:
        z=_dash_norm(token)
        for c,n in normalized.items():
            if z and z in n:
                return c
    return None

def _parse_var_number(series):
    s=series.astype("string").str.strip().str.replace("\u00a0"," ",regex=False)
    s=s.str.replace(" ","",regex=False)
    both=s.str.contains(",",na=False) & s.str.contains(r"\.",na=False)
    s.loc[both]=s.loc[both].str.replace(".","",regex=False).str.replace(",",".",regex=False)
    comma_only=s.str.contains(",",na=False) & ~s.str.contains(r"\.",na=False)
    s.loc[comma_only]=s.loc[comma_only].str.replace(",",".",regex=False)
    s=s.str.replace(r"^[<>≤≥~]+","",regex=True)
    return pd.to_numeric(s,errors="coerce")

def _choose_value_col(d,excluded=()):
    excluded={x for x in excluded if x is not None}
    scored=[]
    for c in d.columns:
        if c in excluded or str(c).startswith("req__") or str(c).startswith("_"):
            continue
        n=_dash_norm(c)
        penalty=0
        if any(x in n for x in ["aandeel","share","percentage","procent","ratio","index","rang"]):
            penalty-=150
        if any(x in n for x in ["nis","code","jaar","year","gemeente","naam","name"]):
            penalty-=100
        bonus=0
        if "aantal" in n or "number" in n:
            bonus+=150
        if any(x in n for x in ["werkenden","personen","jobs","tewerk","employment","loontrekk"]):
            bonus+=80
        if any(x in n for x in ["waarde","value","measure values","度量值"]):
            bonus+=40
        sample=_parse_var_number(d[c].head(5000))
        frac=float(sample.notna().mean()) if len(sample) else 0.0
        if frac<0.25:
            continue
        nonneg=float((sample.dropna()>=0).mean()) if sample.notna().any() else 0.0
        scored.append((bonus+penalty+50*frac+10*nonneg,c))
    if not scored:
        return None
    scored.sort(reverse=True,key=lambda x:x[0])
    return scored[0][1]

# Canonical aliases
_alias_rows=[]
for c in [x for x in ["nis","node_id","var_name","name_nl","name_fr"] if x in geo.columns]:
    for nis,val in zip(geo["nis"],geo[c]):
        if pd.notna(val) and str(val).strip():
            _alias_rows.append((_dash_norm(val),nis))
_alias_df=pd.DataFrame(_alias_rows,columns=["alias","nis"])
_alias_counts=_alias_df.groupby("alias")["nis"].nunique()
GEO_ALIAS={
    a:g["nis"].iloc[0]
    for a,g in _alias_df.groupby("alias")
    if _alias_counts.get(a,0)==1
}
_CANONICAL_NIS=set(geo["nis"])

def _map_var_municipality(v):
    if pd.isna(v):
        return np.nan
    s=str(v).strip()
    m=re.search(r"(?<!\d)(\d{5})(?!\d)",s)
    if m:
        x=to_2025_nis(m.group(1))
        if x in _CANONICAL_NIS:
            return x
    return GEO_ALIAS.get(_dash_norm(s),np.nan)

def _filter_raw_dimension(d,col,preferred=None,label="dimension"):
    if col is None or col not in d.columns:
        return d
    vals=d[col].dropna().astype(str)
    if vals.empty:
        return d
    uniq=list(pd.unique(vals))

    if preferred is not None:
        hit=[v for v in uniq if _value_matches(v,preferred)]
        if hit:
            return d[d[col].astype(str).map(lambda x:_value_matches(x,hit[0]))].copy()

    total=[v for v in uniq if _is_totalish(v)]
    if total:
        return d[d[col].astype(str).map(_is_totalish)].copy()

    if len(uniq)==1:
        return d

    raise ValueError(
        f"{label}: ambiguous nuisance categories with no preferred/total value: {uniq[:15]}"
    )

def _collapse_source_cells_strict(x):
    """
    Collapse exact/identical duplicate rows at source geography level.
    Non-identical duplicates indicate an unfiltered hidden dimension and stop
    the extraction instead of being silently added together.
    """
    rows=[]
    for key,g in x.groupby(
        ["year","source_geo","category"],dropna=False,observed=True
    ):
        vals=pd.to_numeric(g["value"],errors="coerce").dropna()
        if vals.empty:
            continue
        unique=np.sort(vals.unique())
        if len(unique)>1:
            raise ValueError(
                "Non-identical duplicate values remain for "
                f"year={key[0]}, source_geo={key[1]}, category={key[2]}: "
                f"{unique[:8].tolist()}"
            )
        rows.append({
            "year":int(key[0]),
            "source_geo":str(key[1]),
            "category":str(key[2]),
            "value":float(unique[0]),
        })
    return pd.DataFrame(rows)

def extract_direct_heterogeneity_dimension(spec):
    chosen=select_direct_heterogeneity_tasks(spec)
    audit=[]
    pieces=[]

    if chosen.empty:
        audit.append({
            "key":spec["key"],"status":"no_usable_final_plan_task",
            "view":spec["view"],"request_id":None,"year":None
        })
        return pd.DataFrame(),pd.DataFrame(audit),chosen

    nuisance_candidates={
        "sex":("Geslacht","Geslacht_titel","Sex"),
        "education":("Onderwijsniveau","EDU2","EDU_titel","Education"),
        "sector":("WSE42_naam","Sector_titel","Sector","NACE"),
        "nationality":("Nationaliteit","Nationaliteitsklasse"),
        "origin":("Herkomst","Origine Beperkt","Origine"),
        "generation":("Generatie",),
        "work_regime":("Arbeidsregime",),
        "age":("Leeftijdsklasse","Age"),
        "employment_status":("Statuut","Arbeidsmarktpositie"),
        "geography_level":("Geografisch niveau",),
    }

    for row in chosen.itertuples(index=False):
        try:
            d=_read_var_task(row)
            raw_rows=len(d)
            params=_parse_requested_params(row.requested_parameters_json)

            # Municipal geography level if present in returned table.
            level_col=_find_raw_col(
                d.columns,candidates=nuisance_candidates["geography_level"]
            )
            if level_col is not None:
                z=d[level_col].astype(str).map(_dash_norm)
                municipal=z.str.contains("gemeente|municip",regex=True,na=False)
                if municipal.any():
                    d=d[municipal].copy()

            # Filter all nuisance dimensions. The target margin itself is kept.
            target_col=_find_raw_col(d.columns,candidates=spec["target_names"])
            for role,cands in nuisance_candidates.items():
                col=_find_raw_col(d.columns,candidates=cands)
                if col is None:
                    continue
                if role==spec["dimension"] and col==target_col:
                    continue
                if role=="geography_level":
                    continue
                if role=="age":
                    d=_filter_raw_dimension(
                        d,col,spec.get("preferred_age"),f"{spec['key']} age"
                    )
                elif role=="employment_status":
                    d=_filter_raw_dimension(
                        d,col,spec.get("preferred_status"),f"{spec['key']} status"
                    )
                else:
                    d=_filter_raw_dimension(
                        d,col,None,f"{spec['key']} nuisance {role}"
                    )

            if d.empty:
                raise ValueError("No rows remain after nuisance-dimension filtering.")

            muni_col=_find_raw_col(
                d.columns,
                candidates=("NIS","NIS-code","NIS code","Gemeente","Woonplaats","Werkplaats"),
                contains=("nis","gemeente","woonplaats","werkplaats")
            )
            require(muni_col is not None,
                    f"{spec['key']}: no municipality column in {list(d.columns)[:30]}")

            year_col=_find_raw_col(d.columns,candidates=("Jaar","jaar","Year","year"))
            if year_col is not None:
                d["year"]=pd.to_numeric(d[year_col],errors="coerce").astype("Int64")
            else:
                d["year"]=row.task_year

            d=d[d["year"].isin(YEARS)].copy()
            require(len(d)>0,f"{spec['key']}: selected task has no requested years.")

            # Recover target category either from returned data or from the
            # explicit final-plan request parameter.
            request_category=row.target_request_value
            if (
                row.target_mode=="request_parameter"
                and request_category is not None
                and str(request_category).strip()!=""
            ):
                if target_col is not None:
                    visible=d[target_col].dropna().astype(str).unique().tolist()
                    # Keep returned labels if they are present; otherwise the
                    # request parameter is authoritative for this task.
                    if len(visible)==1:
                        d["category"]=str(visible[0]).strip()
                    else:
                        d["category"]=str(request_category).strip()
                else:
                    d["category"]=str(request_category).strip()
            else:
                require(
                    target_col is not None,
                    f"{spec['key']}: target category is neither in request parameters nor raw CSV."
                )
                d["category"]=d[target_col].astype(str).str.strip()

            value_col=_choose_value_col(
                d,excluded={target_col,muni_col,year_col,"year","category"}
            )
            require(value_col is not None,
                    f"{spec['key']}: no plausible count/value column.")

            d["value"]=_parse_var_number(d[value_col])
            d["source_geo"]=d[muni_col].astype(str).str.strip()
            d=d[
                d["category"].ne("")
                & d["value"].notna()
                & d["source_geo"].ne("")
            ].copy()
            require(len(d)>0,f"{spec['key']}: no valid category/value/geography rows.")

            source=_collapse_source_cells_strict(
                d[["year","source_geo","category","value"]]
            )
            source["nis"]=source["source_geo"].map(_map_var_municipality)
            unmatched=int(source["nis"].isna().sum())
            source=source[source["nis"].notna()].copy()
            require(len(source)>0,f"{spec['key']}: no source municipality mapped to canonical NIS.")

            # Sum only across distinct historical source municipalities that
            # legitimately harmonise into one 2025 municipality.
            mapped=(
                source.groupby(["year","nis","category"],as_index=False,observed=True)["value"]
                .sum(min_count=1)
            )
            mapped["side"]=spec["side"]
            mapped["dimension"]=spec["dimension"]
            mapped["source_view"]=spec["view"]
            mapped["request_id"]=row.request_id
            mapped["target_mode"]=row.target_mode
            mapped["category_slug"]=mapped["category"].map(ascii_slug)
            pieces.append(mapped)

            audit.append({
                "key":spec["key"],
                "view":spec["view"],
                "status":"ok",
                "request_id":row.request_id,
                "task_year":row.task_year,
                "target_mode":row.target_mode,
                "target_request_value":row.target_request_value,
                "scope_penalty":row.scope_penalty,
                "raw_rows":raw_rows,
                "kept_rows":len(d),
                "mapped_rows":len(mapped),
                "unmatched_source_geographies":unmatched,
                "target_col":target_col,
                "municipality_col":muni_col,
                "value_col":value_col,
            })

        except Exception as exc:
            audit.append({
                "key":spec["key"],
                "view":spec["view"],
                "status":"error:"+type(exc).__name__,
                "request_id":getattr(row,"request_id",None),
                "task_year":getattr(row,"task_year",None),
                "target_mode":getattr(row,"target_mode",None),
                "target_request_value":getattr(row,"target_request_value",None),
                "scope_penalty":getattr(row,"scope_penalty",None),
                "error":str(exc)[:1000],
            })

    if not pieces:
        return pd.DataFrame(),pd.DataFrame(audit),chosen

    long=pd.concat(pieces,ignore_index=True)

    # De-duplicate overlapping selected responses. Distinct values for the same
    # final cell are not silently combined.
    cell_rows=[]
    for key,g in long.groupby(
        ["year","nis","category"],dropna=False,observed=True
    ):
        vals=pd.to_numeric(g["value"],errors="coerce").dropna()
        if vals.empty:
            continue
        unique=np.sort(vals.unique())
        if len(unique)>1:
            raise ValueError(
                f"{spec['key']}: conflicting final-plan values for "
                f"{key}: {unique[:8].tolist()}"
            )
        cell_rows.append({
            "year":int(key[0]),
            "nis":str(key[1]),
            "category":str(key[2]),
            "value":float(unique[0]),
        })
    long=pd.DataFrame(cell_rows)
    long["side"]=spec["side"]
    long["dimension"]=spec["dimension"]
    long["source_view"]=spec["view"]
    long["category_slug"]=long["category"].map(ascii_slug)

    totals=(
        long[long["category"].map(_is_totalish)]
        .groupby(["year","nis"],as_index=False)["value"].max()
        .rename(columns={"value":"published_total"})
    )
    long=long.merge(
        totals,on=["year","nis"],how="left",validate="many_to_one"
    )
    long["share"]=long["value"]/long["published_total"].replace(0,np.nan)
    long.loc[long["category"].map(_is_totalish),"share"]=1.0

    return long,pd.DataFrame(audit),chosen

def heterogeneity_long_to_wide(long,spec):
    if long is None or long.empty:
        return pd.DataFrame(columns=["year","nis"])

    x=long[~long["category"].map(_is_totalish)].copy()
    prefix=spec["key"]

    count=x.pivot_table(
        index=["year","nis"],columns="category_slug",
        values="value",aggfunc="first"
    )
    count.columns=[f"{prefix}_{c}_count" for c in count.columns]

    share=x.pivot_table(
        index=["year","nis"],columns="category_slug",
        values="share",aggfunc="first"
    )
    share.columns=[f"{prefix}_{c}_share" for c in share.columns]

    out=count.join(share,how="outer").reset_index()
    total=(
        long.groupby(["year","nis"],as_index=False)["published_total"].max()
        .rename(columns={"published_total":f"{prefix}_published_total"})
    )
    return out.merge(total,on=["year","nis"],how="outer",validate="one_to_one")

heterogeneity_longs=[]
heterogeneity_wides=[]
heterogeneity_audits=[]
heterogeneity_task_audits=[]
heterogeneity_family_summary=[]

if BUILD_EXTENDED_HETEROGENEITY:
    for spec in HETERO_SPECS:
        print("Extracting heterogeneity dimension:",spec["key"],"|",spec["view"])
        long,audit,chosen=extract_direct_heterogeneity_dimension(spec)

        ok=not long.empty
        categories=int(
            long.loc[~long["category"].map(_is_totalish),"category"].nunique()
        ) if ok else 0
        n2019=int(
            long.loc[long.year.eq(HETERO_BASELINE_YEAR),"nis"].nunique()
        ) if ok else 0

        heterogeneity_family_summary.append({
            "key":spec["key"],
            "side":spec["side"],
            "dimension":spec["dimension"],
            "source_view":spec["view"],
            "rows":len(long),
            "categories_non_total":categories,
            "municipalities_2019":n2019,
            "expected_min_categories":spec["expected_min_categories"],
            "extraction_ok":bool(
                ok
                and categories>=spec["expected_min_categories"]
                and n2019>=int(EXPECTED_N*0.90)
            ),
        })

        if ok:
            heterogeneity_longs.append(long)
            heterogeneity_wides.append(
                heterogeneity_long_to_wide(long,spec)
            )
            write_table(
                long,
                SUPPORT/f"heterogeneity_{spec['key']}_long.parquet"
            )

        if len(audit):
            heterogeneity_audits.append(audit)

        if len(chosen):
            keep=[
                c for c in [
                    "request_id","module","view","status","rows","raw_file",
                    "requested_parameters_json","header_json",
                    "task_year","target_mode","target_request_field",
                    "target_request_value","target_raw_column","scope_penalty"
                ] if c in chosen.columns
            ]
            q=chosen[keep].copy()
            q["heterogeneity_key"]=spec["key"]
            heterogeneity_task_audits.append(q)

heterogeneity_family_summary=pd.DataFrame(heterogeneity_family_summary)

# Hard completeness gate: do not silently continue with only a subset of the
# requested heterogeneity families.
if BUILD_EXTENDED_HETEROGENEITY:
    missing_families=[
        k for k in EXPECTED_HETERO_FAMILIES
        if k not in set(
            heterogeneity_family_summary.loc[
                heterogeneity_family_summary["extraction_ok"],"key"
            ]
        )
    ]
    if missing_families:
        display(heterogeneity_family_summary)
        if heterogeneity_audits:
            display(pd.concat(heterogeneity_audits,ignore_index=True))
        raise ValueError(
            "Extended heterogeneity extraction is incomplete for: "
            + ", ".join(missing_families)
            + ". Inspect qa/heterogeneity_extended_extraction_audit.csv "
              "and qa/heterogeneity_extended_selected_tasks.csv."
        )

heterogeneity_long=(
    pd.concat(heterogeneity_longs,ignore_index=True)
    if heterogeneity_longs else
    pd.DataFrame(columns=[
        "year","nis","category","value","side","dimension",
        "source_view","category_slug","published_total","share"
    ])
)

heterogeneity_wide=pd.DataFrame(columns=["year","nis"])
for q in heterogeneity_wides:
    heterogeneity_wide=(
        q if heterogeneity_wide.empty
        else heterogeneity_wide.merge(
            q,on=["year","nis"],how="outer",validate="one_to_one"
        )
    )

heterogeneity_audit=(
    pd.concat(heterogeneity_audits,ignore_index=True)
    if heterogeneity_audits else pd.DataFrame()
)
heterogeneity_task_audit=(
    pd.concat(heterogeneity_task_audits,ignore_index=True)
    if heterogeneity_task_audits else pd.DataFrame()
)

# Convenience composites. Direct source categories remain available separately.
def _first_col(columns,prefix,tokens):
    for c in columns:
        if not c.startswith(prefix) or not c.endswith("_share"):
            continue
        z=_dash_norm(c)
        if all(t in z for t in tokens):
            return c
    return None

for side in ("resident","workplace"):
    c=_first_col(
        heterogeneity_wide.columns,
        f"{side}_nationality_",("belgie",)
    )
    if c:
        heterogeneity_wide[f"{side}_foreign_nationality_share"]=1-heterogeneity_wide[c]

    c=_first_col(
        heterogeneity_wide.columns,
        f"{side}_origin_",("geen","vreemde","herkomst")
    )
    if c:
        heterogeneity_wide[f"{side}_foreign_origin_share"]=1-heterogeneity_wide[c]
    else:
        c=_first_col(
            heterogeneity_wide.columns,
            f"{side}_origin_",("belgie",)
        )
        if c:
            heterogeneity_wide[f"{side}_foreign_origin_share"]=1-heterogeneity_wide[c]

heterogeneity_share_cols=[
    c for c in heterogeneity_wide.columns
    if c not in ("year","nis") and c.endswith("_share")
]

heterogeneity_baseline=pd.DataFrame({"nis":geo["nis"].astype(str)})
for c in heterogeneity_share_cols:
    b=(
        heterogeneity_wide.loc[
            heterogeneity_wide.year.eq(HETERO_BASELINE_YEAR),["nis",c]
        ]
        .drop_duplicates("nis")
        .rename(columns={c:f"{c}_{HETERO_BASELINE_YEAR}"})
    )
    heterogeneity_baseline=heterogeneity_baseline.merge(
        b,on="nis",how="left",validate="one_to_one"
    )

    pre=(
        heterogeneity_wide.loc[
            heterogeneity_wide.year.isin(PRE_YEARS),["nis",c]
        ]
        .groupby("nis")[c]
        .agg(["mean","count"])
        .reset_index()
        .rename(columns={
            "mean":f"{c}_premean_available_pre2020",
            "count":f"{c}_pre_n_years",
        })
    )
    heterogeneity_baseline=heterogeneity_baseline.merge(
        pre,on="nis",how="left",validate="one_to_one"
    )

# Dictionary
dict_rows=[]
for spec in HETERO_SPECS:
    prefix=spec["key"]+"_"
    for c in [x for x in heterogeneity_wide.columns if x.startswith(prefix)]:
        dict_rows.append({
            "variable":c,
            "side":spec["side"],
            "dimension":spec["dimension"],
            "source_view":spec["view"],
            "preferred_age":spec.get("preferred_age"),
            "preferred_status":spec.get("preferred_status"),
            "role":"heterogeneity_candidate",
            "causal_note":(
                "Use the 2019 baseline version for main causal heterogeneity; "
                "contemporaneous annual composition may respond to the post-2020 shock."
            ),
        })
heterogeneity_dictionary=pd.DataFrame(dict_rows)

coverage_rows=[]
if len(heterogeneity_long):
    for (side,dimension,year),g in heterogeneity_long.groupby(
        ["side","dimension","year"],observed=True
    ):
        coverage_rows.append({
            "side":side,
            "dimension":dimension,
            "year":int(year),
            "municipalities_with_any_data":int(g["nis"].nunique()),
            "categories":int(g["category"].nunique()),
            "categories_non_total":int(
                g.loc[~g["category"].map(_is_totalish),"category"].nunique()
            ),
            "municipalities_with_total":int(
                g.loc[g["published_total"].notna(),"nis"].nunique()
            ),
        })
heterogeneity_coverage=pd.DataFrame(coverage_rows)

# Exports
write_table(
    heterogeneity_long,
    SUPPORT/"heterogeneity_extended_long.parquet"
)
write_table(
    heterogeneity_wide,
    SUPPORT/"heterogeneity_extended_municipality_year.parquet"
)
write_table(
    heterogeneity_baseline,
    SUPPORT/"heterogeneity_extended_baseline.parquet"
)
heterogeneity_dictionary.to_csv(
    SUPPORT/"heterogeneity_extended_variable_dictionary.csv",
    index=False,encoding="utf-8-sig"
)
heterogeneity_family_summary.to_csv(
    QA_DIR/"heterogeneity_extended_family_summary.csv",
    index=False,encoding="utf-8-sig"
)
heterogeneity_coverage.to_csv(
    QA_DIR/"heterogeneity_extended_coverage_by_year.csv",
    index=False,encoding="utf-8-sig"
)
if len(heterogeneity_audit):
    heterogeneity_audit.to_csv(
        QA_DIR/"heterogeneity_extended_extraction_audit.csv",
        index=False,encoding="utf-8-sig"
    )
if len(heterogeneity_task_audit):
    heterogeneity_task_audit.to_csv(
        QA_DIR/"heterogeneity_extended_selected_tasks.csv",
        index=False,encoding="utf-8-sig"
    )

# Numerical validity
if heterogeneity_share_cols:
    vals=heterogeneity_wide[heterogeneity_share_cols].apply(
        pd.to_numeric,errors="coerce"
    ).stack()
    require(
        ((vals>=-1e-9)&(vals<=1+1e-9)).all(),
        "An extended heterogeneity share falls outside [0,1]."
    )
if len(heterogeneity_long):
    require(
        set(heterogeneity_long["nis"].dropna()).issubset(_CANONICAL_NIS),
        "Extended heterogeneity contains NIS codes outside the canonical geography."
    )

print("Extended heterogeneity long:",heterogeneity_long.shape)
print("Extended heterogeneity wide:",heterogeneity_wide.shape)
print("2019 baseline columns:",sum(c.endswith("_2019") for c in heterogeneity_baseline.columns))
print("\nHeterogeneity family QA:")
display(heterogeneity_family_summary)
if len(heterogeneity_coverage):
    display(heterogeneity_coverage.sort_values(["side","dimension","year"]))


## 9. Sector structure and WSE42 long tables

In [ ]:

WSE_TO_NACE = {
    "p1":"A;B",
    **{f"s{i}":"C" for i in range(1,14)},
    "s14":"D;E","s15":"E","s16":"F",
    "t1":"C;S","t2":"G","t3":"G","t4":"G","t5":"H","t6":"H","t7":"H",
    "t8":"I;N","t9":"J","t10":"J","t11":"J","t12":"K","t13":"M","t14":"N","t15":"N",
    "t16":"L;N","t17":"M;S;T",
    "q1":"R","q2":"O","q3":"O;U","q4":"O","q5":"P","q6":"Q","q7":"Q","q8":"S"
}
WSE_CORE_SINGLE_NACE = {k:v for k,v in WSE_TO_NACE.items() if ";" not in v}

sector_raw = load_od(OD_FILES["sector"],dimension="sector",age_scope="20-64")
sector_raw["sector_code"] = sector_raw["sector"].astype(str).str.extract(r"^([pqst]\d+)\b",flags=re.I)[0].str.lower()
sector_raw = sector_raw[sector_raw.sector_code.isin(WSE_TO_NACE)].copy()

sector_parts = []
for side,nodecol in [("residence","home_nis"),("workplace","work_nis")]:
    q = (
        sector_raw[sector_raw[nodecol].notna()]
        .groupby(["year",nodecol,"sector_code"],as_index=False)["workers"].sum(min_count=1)
        .rename(columns={nodecol:"nis"})
        .assign(side=side)
    )
    sector_parts.append(q)
sector_long = pd.concat(sector_parts,ignore_index=True)

main_side_total = pd.concat([
    resident_workers.rename(columns={"resident_workers":"published_total"}).assign(side="residence"),
    workplace_jobs.rename(columns={"workplace_jobs":"published_total"}).assign(side="workplace"),
],ignore_index=True)
sector_long = sector_long.merge(main_side_total,on=["side","year","nis"],how="left",validate="many_to_one")
sector_long["share_of_main_total"] = sector_long["workers"]/sector_long["published_total"].replace(0,np.nan)
write_table(sector_long,SUPPORT/"06_sector_panel_residence_workplace.parquet")
write_table(sector_long[sector_long.side.eq("residence")].drop(columns="side"),
            SUPPORT/"06_sector_panel_residence.parquet")
write_table(sector_long[sector_long.side.eq("workplace")].drop(columns="side"),
            SUPPORT/"07_sector_panel_workplace.parquet")

# Compact municipality-year sector summaries for the master panel.
sector_summary_rows = []
for (side,year,nis),g in sector_long.groupby(["side","year","nis"],observed=True):
    total = g["published_total"].dropna()
    total = float(total.iloc[0]) if len(total) else np.nan
    known = g["workers"].sum(min_count=1)
    w = g["workers"].to_numpy(float)
    wf = w[np.isfinite(w) & (w >= 0)]
    p = wf/wf.sum() if len(wf) and wf.sum()>0 else np.array([])
    row = {
        "side":side,"year":year,"nis":nis,"sector_known_workers":known,
        "sector_coverage_vs_main":known/total if pd.notna(total) and total>0 else np.nan,
        "sector_hhi_known":float(np.sum(p*p)) if len(p) else np.nan,
        "sector_entropy_known":float(-np.sum(p[p>0]*np.log(p[p>0]))) if len(p) else np.nan,
    }
    for macro,prefix in [("primary","p"),("secondary","s"),("tertiary","t"),("quaternary","q")]:
        m = g.loc[g.sector_code.str.startswith(prefix),"workers"].sum(min_count=1)
        row[macro+"_share_main"] = m/total if pd.notna(total) and total>0 else np.nan
    sector_summary_rows.append(row)
sector_summary = pd.DataFrame(sector_summary_rows)

res_sector_summary = sector_summary[sector_summary.side.eq("residence")].drop(columns="side").rename(
    columns={c:"resident_"+c for c in sector_summary.columns if c not in ["side","year","nis"]})
work_sector_summary = sector_summary[sector_summary.side.eq("workplace")].drop(columns="side").rename(
    columns={c:"workplace_"+c for c in sector_summary.columns if c not in ["side","year","nis"]})
sector_summary_wide = res_sector_summary.merge(work_sector_summary,on=["year","nis"],how="outer",validate="one_to_one")
write_table(sector_summary_wide,SUPPORT/"sector_structure_summary.parquet")


## 10. Network outcomes and node-level topology — residence and workplace sides

In [ ]:

def one_node_network(g, node_col, partner_col):
    w = g["workers"].to_numpy(float)
    d = g["distance_km"].to_numpy(float)
    node = str(g[node_col].iloc[0])
    partner = g[partner_col].astype(str).to_numpy()
    ext = partner != node
    positive = np.isfinite(w) & (w>0)
    total = np.nansum(w)
    cross = np.nansum(w[ext])
    local = np.nansum(w[~ext])
    shares = w[positive]/np.nansum(w[positive]) if positive.any() else np.array([])
    hhi = np.sum(shares**2) if len(shares) else np.nan
    entropy = -np.sum(shares*np.log(shares)) if len(shares) else np.nan
    sorted_shares = np.sort(shares)[::-1] if len(shares) else np.array([])
    q50,q75,q90 = weighted_quantile(d,w,(.5,.75,.9))
    de = d[ext]; we = w[ext]
    e50,e75,e90 = weighted_quantile(de,we,(.5,.75,.9))
    return pd.Series({
        "workers":total,
        "local_workers":local,
        "cross_workers":cross,
        "local_share":local/total if total>0 else np.nan,
        "cross_share":cross/total if total>0 else np.nan,
        "mean_km":np.nansum(w*d)/total if total>0 else np.nan,
        "external_mean_km":np.nansum(we*de)/cross if cross>0 else np.nan,
        "long30_share":np.nansum(w[d>30])/total if total>0 else np.nan,
        "long50_share":np.nansum(w[d>50])/total if total>0 else np.nan,
        "external_long30_share":np.nansum(we[de>30])/cross if cross>0 else np.nan,
        "external_long50_share":np.nansum(we[de>50])/cross if cross>0 else np.nan,
        "p50_km":q50,"p75_km":q75,"p90_km":q90,
        "external_p50_km":e50,"external_p75_km":e75,"external_p90_km":e90,
        "n_published_links":len(g),
        "n_positive_links":int(positive.sum()),
        "external_positive_partners":int((positive & ext).sum()),
        "flow_hhi":hhi,
        "flow_entropy":entropy,
        "effective_partners":float(np.exp(entropy)) if pd.notna(entropy) else np.nan,
        "top1_flow_share":sorted_shares[0] if len(sorted_shares) else np.nan,
        "top3_flow_share":sorted_shares[:3].sum() if len(sorted_shares) else np.nan,
    })

def _node_metric_table(od, node_col, partner_col, prefix):
    rows = []
    for (year, node), g in od.groupby(
        ["year", node_col], observed=True, sort=False
    ):
        rec = one_node_network(g, node_col, partner_col).to_dict()
        rec["year"] = int(year)
        rec["nis"] = str(node)
        rows.append(rec)
    out = pd.DataFrame(rows)
    front = ["year","nis"]
    out = out[front + [c for c in out.columns if c not in front]]
    out = out.rename(
        columns={c:prefix+c for c in out.columns if c not in front}
    )
    return out.sort_values(["nis","year"]).reset_index(drop=True)

def build_node_network_metrics(od):
    # Explicit iteration is stable across pandas versions and avoids
    # DataFrameGroupBy.apply grouping-column deprecation warnings.
    res = _node_metric_table(
        od, "home_nis", "work_nis", "resnet_"
    )
    work = _node_metric_table(
        od, "work_nis", "home_nis", "worknet_"
    )
    return res, work

resnet, worknet = build_node_network_metrics(od_domestic)
write_table(resnet,SUPPORT/"04_network_metrics_residence.parquet")
write_table(worknet,SUPPORT/"04_network_metrics_workplace.parquet")
print("Residence network panel:",resnet.shape,"Workplace network panel:",worknet.shape)


## 11. Accessibility from the same 565-municipality distance system

In [ ]:

id_order = geo["nis"].tolist()
D = pair.pivot(index="home_nis",columns="work_nis",values="distance_km").reindex(index=id_order,columns=id_order).to_numpy(float)
W_EXP50 = np.exp(-D/50.0)
W_EXP100 = np.exp(-D/100.0)
W_INV = 1.0/(1.0+D)
np.fill_diagonal(W_EXP50,1.0)
np.fill_diagonal(W_EXP100,1.0)
np.fill_diagonal(W_INV,1.0)

def accessibility_from_panel(panel, value_col, prefix):
    rows = []
    for y in YEARS:
        s = panel[panel.year.eq(y)].set_index("nis")[value_col].reindex(id_order)
        x = s.to_numpy(float)
        valid = np.isfinite(x)
        if not valid.any():
            continue
        for name,W in [("exp50",W_EXP50),("exp100",W_EXP100),("inv",W_INV)]:
            num = W[:,valid] @ x[valid]
            # Not row-normalised: this is Hansen-style potential accessibility.
            if name=="exp50":
                vals50 = num
            elif name=="exp100":
                vals100 = num
            else:
                valsinv = num
        rows.append(pd.DataFrame({
            "year":y,"nis":id_order,
            prefix+"_access_exp50":vals50,
            prefix+"_access_exp100":vals100,
            prefix+"_access_inv":valsinv,
        }))
    out = pd.concat(rows,ignore_index=True)
    for c in [prefix+"_access_exp50",prefix+"_access_exp100",prefix+"_access_inv"]:
        out["ln_"+c] = np.log(out[c].where(out[c]>0))
    return out

job_access = accessibility_from_panel(employment[["year","nis","workplace_jobs"]],"workplace_jobs","job")
pop_access = accessibility_from_panel(population[["year","nis","population"]],"population","population")
accessibility = job_access.merge(pop_access,on=["year","nis"],how="outer",validate="one_to_one")
write_table(accessibility,SUPPORT/"accessibility_municipality_year.parquet")


## 12. Telework/WFH exposures: current, frozen-share, Bartik, and 2018/2019 baseline variants

In [ ]:

def load_wfh_rates():
    if WFH_RATE_CSV.exists():
        r = pd.read_csv(WFH_RATE_CSV,encoding="utf-8-sig")
        need = {"year","nace_section","any_wfh_rate"}
        require(need.issubset(r.columns), f"{WFH_RATE_CSV} missing {need-set(r.columns)}")
        r["year"] = pd.to_numeric(r["year"],errors="coerce").astype("Int64")
        r["nace_section"] = r["nace_section"].astype(str).str.strip().str.upper()
        r["any_wfh_rate"] = pd.to_numeric(r["any_wfh_rate"],errors="coerce")
        r = r[r.nace_section.str.fullmatch(r"[A-U]",na=False)].copy()
        require(((r.any_wfh_rate.dropna()>=0)&(r.any_wfh_rate.dropna()<=1)).all(),"WFH rate outside [0,1]")
        return r
    warnings.warn(
        "Verified NACE-year WFH rate CSV not found. Exposure construction is skipped. "
        "Run the earlier telework builder or point WFH_RATE_CSV to its 01_nace_year_wfh_rates_2010_2025.csv output."
    )
    return pd.DataFrame()

rates = load_wfh_rates()

def build_current_sector_exposure(sector_long, rates):
    if rates.empty:
        return pd.DataFrame()
    mapdf = pd.DataFrame([{"sector_code":k,"nace_section":v} for k,v in WSE_CORE_SINGLE_NACE.items()])
    x = sector_long.merge(mapdf,on="sector_code",how="left",validate="many_to_one")
    x = x.merge(rates[["year","nace_section","any_wfh_rate"]],on=["year","nace_section"],how="left",validate="many_to_one")
    x["supported_worker_rate"] = x["workers"]*x["any_wfh_rate"]
    rows = []
    for (side,year,nis),g in x.groupby(["side","year","nis"],observed=True):
        total = g["published_total"].dropna()
        total = float(total.iloc[0]) if len(total) else np.nan
        core = g[g.nace_section.notna() & g.any_wfh_rate.notna()].copy()
        core_workers = core["workers"].sum(min_count=1)
        value = core["supported_worker_rate"].sum(min_count=1)/core_workers if pd.notna(core_workers) and core_workers>0 else np.nan
        rows.append({
            "side":side,"year":year,"nis":nis,
            "tw_current":value,
            "tw_core_workers":core_workers,
            "tw_core_coverage_total":core_workers/total if pd.notna(total) and total>0 else np.nan,
        })
    return pd.DataFrame(rows)

def _nace_worker_matrix(sector_long, side):
    mapdf = pd.DataFrame([{"sector_code":k,"nace_section":v} for k,v in WSE_CORE_SINGLE_NACE.items()])
    x = sector_long[sector_long.side.eq(side)].merge(mapdf,on="sector_code",how="inner",validate="many_to_one")
    return x.groupby(["nis","year","nace_section"],as_index=False)["workers"].sum(min_count=1)

def build_frozen_and_bartik(sector_long, rates, side_value, prefix, share_year=2019):
    x = _nace_worker_matrix(sector_long,side_value)
    if x.empty or rates.empty:
        return pd.DataFrame()
    base = x[x.year.eq(share_year)].copy()
    base = base.merge(rates[rates.year.eq(share_year)][["nace_section","any_wfh_rate"]]
                      .rename(columns={"any_wfh_rate":"rate_base"}),on="nace_section",how="left",validate="many_to_one")
    base = base[base.rate_base.notna()].copy()
    den = base.groupby("nis")["workers"].transform("sum")
    base["w_share"] = base["workers"]/den.replace(0,np.nan)
    base_exp = (base.assign(z=base.w_share*base.rate_base).groupby("nis",as_index=False)["z"].sum()
                .rename(columns={"z":f"tw_{prefix}_s{share_year}_r{share_year}"}))
    rows = []
    for y in sorted(set(YEARS)&set(pd.to_numeric(rates.year,errors="coerce").dropna().astype(int))):
        rr = rates[rates.year.eq(y)][["nace_section","any_wfh_rate"]]
        q = base.merge(rr,on="nace_section",how="left",validate="many_to_one")
        q["weighted_rate"] = q["w_share"]*q["any_wfh_rate"]
        z = q.groupby("nis").agg(
            frozen=("weighted_rate","sum"),
            positive_weight=("w_share","sum"),
            missing_rate_on_positive_weight=("any_wfh_rate",lambda s:int(s.isna().sum()))
        ).reset_index()
        z["year"] = y
        value_col = f"tw_{prefix}_frozen{share_year}"
        cov_col = f"{value_col}_weight_coverage"
        miss_col = f"{value_col}_missing_rate_cells"
        z[value_col] = z["frozen"].where(z.missing_rate_on_positive_weight.eq(0))
        z[cov_col] = z["positive_weight"]
        z[miss_col] = z["missing_rate_on_positive_weight"]
        rows.append(z[["year","nis",value_col,cov_col,miss_col]])
    out = pd.concat(rows,ignore_index=True)
    out = out.merge(base_exp,on="nis",how="left",validate="many_to_one")
    basecol = f"tw_{prefix}_s{share_year}_r{share_year}"
    out[f"bartik_{prefix}_{share_year}"] = out[f"tw_{prefix}_frozen{share_year}"]-out[basecol]
    return out

def baseline_2x2(sector_long,rates,side_value,prefix):
    x = _nace_worker_matrix(sector_long,side_value)
    if x.empty or rates.empty:
        return pd.DataFrame()
    p = x[x.year.isin([2018,2019])].pivot_table(index=["nis","nace_section"],columns="year",values="workers",aggfunc="sum")
    p = p.reset_index()
    for y in (2018,2019):
        if y not in p:
            p[y] = np.nan
    rr = rates[rates.year.isin([2018,2019])].pivot(index="nace_section",columns="year",values="any_wfh_rate")
    p["r2018"] = p.nace_section.map(rr[2018] if 2018 in rr else pd.Series(dtype=float))
    p["r2019"] = p.nace_section.map(rr[2019] if 2019 in rr else pd.Series(dtype=float))
    p["support"] = p[2018].notna() & p[2019].notna() & p.r2018.notna() & p.r2019.notna()
    p = p[p.support].copy()
    rows = []
    for nis,g in p.groupby("nis"):
        row={"nis":nis,f"tw_{prefix}_shared_core_cells":len(g)}
        for sy in (2018,2019):
            den = g[sy].sum()
            if den<=0:
                continue
            w = g[sy]/den
            for ry in (2018,2019):
                row[f"tw_{prefix}_s{sy}_r{ry}"] = float(np.sum(w*g[f"r{ry}"]))
        rows.append(row)
    return pd.DataFrame(rows)

if not rates.empty:
    current = build_current_sector_exposure(sector_long,rates)
    current_res = current[current.side.eq("residence")].drop(columns="side").rename(
        columns={"tw_current":"tw_res_current","tw_core_workers":"tw_res_core_workers",
                 "tw_core_coverage_total":"tw_res_core_coverage_total"})
    current_work = current[current.side.eq("workplace")].drop(columns="side").rename(
        columns={"tw_current":"tw_work_current","tw_core_workers":"tw_work_core_workers",
                 "tw_core_coverage_total":"tw_work_core_coverage_total"})
    exposure_panel = current_res.merge(current_work,on=["year","nis"],how="outer",validate="one_to_one")
    exposure_panel = exposure_panel.merge(build_frozen_and_bartik(sector_long,rates,"residence","res",2019),on=["year","nis"],how="outer",validate="one_to_one")
    exposure_panel = exposure_panel.merge(build_frozen_and_bartik(sector_long,rates,"workplace","work",2019),on=["year","nis"],how="outer",validate="one_to_one")
    base_res = baseline_2x2(sector_long,rates,"residence","res")
    base_work = baseline_2x2(sector_long,rates,"workplace","work")
    base = base_res.merge(base_work,on="nis",how="outer",validate="one_to_one")
    exposure_panel = exposure_panel.merge(base,on="nis",how="left",validate="many_to_one")
else:
    exposure_panel = pd.MultiIndex.from_product([YEARS,geo.nis],names=["year","nis"]).to_frame(index=False)

# Optional preferred workplace baseline from the verified formal workplace-margin component.
official_work_baseline = pd.DataFrame()
if not rates.empty and WFH_WORKPLACE_MARGIN_CSV.exists():
    m = pd.read_csv(WFH_WORKPLACE_MARGIN_CSV,encoding="utf-8-sig")
    m["year"] = pd.to_numeric(m["year"],errors="coerce").astype("Int64")
    if "nis" not in m.columns and "node_id" in m.columns:
        m["nis"] = m["node_id"].map(NODE_TO_NIS)
    m["nis"] = m["nis"].map(nis5)
    m["wse_code"] = m["wse_code"].astype(str).str.lower().str.strip()
    m["jobs"] = pd.to_numeric(m["jobs"],errors="coerce")
    mm = m[m.wse_code.isin(WSE_CORE_SINGLE_NACE)].copy()
    mm["nace_section"] = mm["wse_code"].map(WSE_CORE_SINGLE_NACE)
    mm = mm.groupby(["nis","year","nace_section"],as_index=False)["jobs"].sum(min_count=1)
    pseudo = mm.rename(columns={"jobs":"workers"}).assign(side="workplace",published_total=np.nan,sector_code="x")
    # Reuse a compact 2x2 implementation directly on NACE rows.
    p = mm[mm.year.isin([2018,2019])].pivot_table(index=["nis","nace_section"],columns="year",values="jobs",aggfunc="sum").reset_index()
    for y in (2018,2019):
        if y not in p: p[y]=np.nan
    rr = rates[rates.year.isin([2018,2019])].pivot(index="nace_section",columns="year",values="any_wfh_rate")
    p["r2018"]=p.nace_section.map(rr[2018]); p["r2019"]=p.nace_section.map(rr[2019])
    p=p[p[2018].notna()&p[2019].notna()&p.r2018.notna()&p.r2019.notna()]
    rec=[]
    for nis,g in p.groupby("nis"):
        row={"nis":nis}
        for sy in (2018,2019):
            den=g[sy].sum()
            if den>0:
                w=g[sy]/den
                for ry in (2018,2019):
                    row[f"tw_work_official_s{sy}_r{ry}"]=float(np.sum(w*g[f"r{ry}"]))
        rec.append(row)
    official_work_baseline=pd.DataFrame(rec)
    exposure_panel=exposure_panel.merge(official_work_baseline,on="nis",how="left",validate="many_to_one")

if len(exposure_panel):
    write_table(exposure_panel,SUPPORT/"05_telework_exposure_municipality_year.parquet")
    if len(official_work_baseline):
        write_table(official_work_baseline,SUPPORT/"telework_official_workplace_baseline.parquet")


### 12.1 Telework exposure coverage audit

No missing municipality exposure is imputed. This audit identifies every municipality-year missing from the principal exposure measures and diagnoses whether the gap originates in the local WSE42 sector table, the single-NACE mapping, national WFH-rate support, or a downstream merge.

The audit is particularly important for the 2019-share/Bartik measures because one missing 2019 municipality propagates to every year of the frozen-share series.

In [ ]:

EXPOSURE_AUDIT_VARS = [
    c for c in [
        "tw_res_current","tw_work_current",
        "bartik_res_2019","bartik_work_2019",
        "tw_res_s2019_r2019","tw_work_s2019_r2019",
        "tw_res_s2018_r2018","tw_work_s2018_r2018",
    ] if c in exposure_panel.columns
]

def _exposure_side_and_reference(variable, observation_year):
    if "_res_" in variable or variable.startswith("tw_res") or variable.startswith("bartik_res"):
        side = "residence"
    elif "_work_" in variable or variable.startswith("tw_work") or variable.startswith("bartik_work"):
        side = "workplace"
    else:
        side = None

    if "bartik_" in variable or "_s2019_" in variable or "frozen2019" in variable:
        sector_year = 2019
    elif "_s2018_" in variable:
        sector_year = 2018
    else:
        sector_year = int(observation_year)

    if "bartik_" in variable:
        rate_year = int(observation_year)
    elif "_r2019" in variable:
        rate_year = 2019
    elif "_r2018" in variable:
        rate_year = 2018
    else:
        rate_year = int(observation_year)

    return side, sector_year, rate_year

def diagnose_missing_exposure(nis, variable, observation_year):
    side, sector_year, rate_year = _exposure_side_and_reference(
        variable, observation_year
    )
    rec = {
        "variable": variable,
        "year": int(observation_year),
        "nis": str(nis),
        "side": side,
        "sector_reference_year": sector_year,
        "rate_reference_year": rate_year,
        "reason": "unclassified",
        "sector_rows": 0,
        "single_nace_sector_rows": 0,
        "single_nace_workers": np.nan,
        "supported_rate_sector_rows": 0,
        "supported_rate_workers": np.nan,
        "published_total": np.nan,
        "supported_worker_share_of_published_total": np.nan,
    }
    if side is None:
        rec["reason"] = "cannot_infer_side"
        return rec

    g = sector_long[
        sector_long.side.eq(side)
        & sector_long.year.eq(sector_year)
        & sector_long.nis.astype(str).eq(str(nis))
    ].copy()
    rec["sector_rows"] = len(g)
    if g.empty:
        rec["reason"] = f"no_{sector_year}_sector_rows"
        return rec

    total = pd.to_numeric(g["published_total"], errors="coerce").dropna()
    rec["published_total"] = float(total.iloc[0]) if len(total) else np.nan

    core = g[g.sector_code.isin(WSE_CORE_SINGLE_NACE)].copy()
    rec["single_nace_sector_rows"] = len(core)
    rec["single_nace_workers"] = pd.to_numeric(
        core["workers"], errors="coerce"
    ).sum(min_count=1)
    if core.empty:
        rec["reason"] = f"no_{sector_year}_single_nace_sector_rows"
        return rec

    mapdf = pd.DataFrame(
        [{"sector_code":k,"nace_section":v} for k,v in WSE_CORE_SINGLE_NACE.items()]
    )
    core = core.merge(mapdf,on="sector_code",how="left",validate="many_to_one")
    rr = rates[rates.year.eq(rate_year)][
        ["nace_section","any_wfh_rate"]
    ].copy()
    core = core.merge(
        rr,on="nace_section",how="left",validate="many_to_one"
    )
    supported = core[core.any_wfh_rate.notna()].copy()
    rec["supported_rate_sector_rows"] = len(supported)
    rec["supported_rate_workers"] = pd.to_numeric(
        supported["workers"], errors="coerce"
    ).sum(min_count=1)

    if (
        pd.notna(rec["published_total"])
        and rec["published_total"] > 0
        and pd.notna(rec["supported_rate_workers"])
    ):
        rec["supported_worker_share_of_published_total"] = (
            rec["supported_rate_workers"] / rec["published_total"]
        )

    if supported.empty:
        rec["reason"] = f"no_wfh_rate_support_for_{rate_year}"
    elif pd.isna(rec["supported_rate_workers"]) or rec["supported_rate_workers"] <= 0:
        rec["reason"] = "nonpositive_supported_sector_workers"
    elif pd.isna(rec["published_total"]):
        rec["reason"] = "supported_sector_rows_but_main_total_missing"
    else:
        rec["reason"] = "unexpected_downstream_missing_after_supported_sector_rows"
    return rec

coverage_rows = []
missing_rows = []
for variable in EXPOSURE_AUDIT_VARS:
    for y in YEARS:
        frame = exposure_panel.loc[
            exposure_panel.year.eq(y), ["nis", variable]
        ].copy()
        # Reindex to the canonical geography so a wholly absent municipality is visible.
        frame = (
            pd.DataFrame({"nis":geo["nis"].astype(str)})
            .merge(frame,on="nis",how="left",validate="one_to_one")
        )
        n_nonmissing = int(frame[variable].notna().sum())
        coverage_rows.append({
            "variable":variable,
            "year":y,
            "n_municipalities":EXPECTED_N,
            "n_nonmissing":n_nonmissing,
            "n_missing":EXPECTED_N-n_nonmissing,
            "share_nonmissing":n_nonmissing/EXPECTED_N,
        })
        for nis in frame.loc[frame[variable].isna(),"nis"]:
            missing_rows.append(
                diagnose_missing_exposure(nis, variable, y)
            )

exposure_coverage_audit = pd.DataFrame(coverage_rows)
exposure_missing_detail = pd.DataFrame(missing_rows)

if not exposure_missing_detail.empty:
    exposure_missing_detail = exposure_missing_detail.merge(
        geo[[c for c in ["nis","name_nl","name_fr","prov_nis","reg_nis"] if c in geo.columns]],
        on="nis", how="left", validate="many_to_one"
    )

exposure_coverage_audit.to_csv(
    QA_DIR/"telework_exposure_coverage_by_year.csv",
    index=False,encoding="utf-8-sig"
)
exposure_missing_detail.to_csv(
    QA_DIR/"telework_exposure_missing_municipalities.csv",
    index=False,encoding="utf-8-sig"
)

baseline_vars = [
    c for c in [
        "tw_res_s2019_r2019","tw_work_s2019_r2019",
        "bartik_res_2019","bartik_work_2019"
    ] if c in exposure_panel.columns
]
if baseline_vars:
    baseline_diag = (
        exposure_coverage_audit[
            exposure_coverage_audit.variable.isin(baseline_vars)
        ]
        .groupby("variable",as_index=False)
        .agg(
            min_nonmissing=("n_nonmissing","min"),
            max_nonmissing=("n_nonmissing","max"),
            total_missing_cells=("n_missing","sum"),
        )
    )
    baseline_diag.to_csv(
        QA_DIR/"telework_exposure_baseline_coverage_summary.csv",
        index=False,encoding="utf-8-sig"
    )
    print("Telework baseline coverage:")
    display(baseline_diag)

if not exposure_missing_detail.empty:
    unique_missing = (
        exposure_missing_detail[
            [c for c in ["nis","name_nl","name_fr","prov_nis","reg_nis","reason"]
             if c in exposure_missing_detail.columns]
        ]
        .drop_duplicates()
        .sort_values(["nis","reason"])
    )
    print("Municipalities with at least one missing audited exposure:")
    display(unique_missing)
else:
    print("No missing values in audited telework exposure variables.")


## 13. Pre-pandemic commuting-network exposure and spatial spillover exposure

In [ ]:

# Pre-pandemic domestic OD weights. Both any-published and stable-2017-2019 versions are retained.
pre = od_domestic[od_domestic.year.isin([2017,2018,2019])].copy()
pre_pair = pre.groupby(["home_nis","work_nis"]).agg(
    pre_flow_sum=("workers","sum"),
    n_pre_years=("year","nunique")
).reset_index()

def add_weights(df, side):
    node = "home_nis" if side=="out" else "work_nis"
    q = df.copy()
    den = q.groupby(node)["pre_flow_sum"].transform("sum")
    q[f"w_pre_{side}_any"] = q["pre_flow_sum"]/den.replace(0,np.nan)
    stable = q.n_pre_years.eq(3)
    stabden = q["pre_flow_sum"].where(stable,0).groupby(q[node]).transform("sum")
    q[f"w_pre_{side}_stable"] = np.where(stable,q["pre_flow_sum"]/stabden.replace(0,np.nan),0.0)
    return q

pre_weights = add_weights(add_weights(pre_pair,"out"),"in")
write_table(pre_weights,SUPPORT/"09_network_weights_pre2019"/"commuting_weights_2017_2019.parquet")

def network_weighted_exposure(exposure_panel, var, direction="out", stable=False):
    if var not in exposure_panel.columns:
        return pd.DataFrame()
    wcol = f"w_pre_{direction}_{'stable' if stable else 'any'}"
    if direction=="out":
        node, other = "home_nis","work_nis"
        prefix = "out"
    else:
        node, other = "work_nis","home_nis"
        prefix = "in"
    w = pre_weights[[node,other,wcol]].copy()
    rows=[]
    for y in YEARS:
        e = exposure_panel[exposure_panel.year.eq(y)][["nis",var]].rename(columns={"nis":other})
        q = w.merge(e,on=other,how="left",validate="many_to_one")
        q["valid_w"] = q[wcol].where(q[var].notna(),0)
        q["wx"] = q[wcol]*q[var]
        a = q.groupby(node).agg(wx=("wx","sum"),coverage=("valid_w","sum")).reset_index()
        a["value"] = a["wx"]/a["coverage"].replace(0,np.nan)
        a["year"]=y
        a=a.rename(columns={node:"nis","value":f"network_{prefix}_{var}_{'stable' if stable else 'any'}",
                            "coverage":f"network_{prefix}_{var}_{'stable' if stable else 'any'}_coverage"})
        rows.append(a[["year","nis",f"network_{prefix}_{var}_{'stable' if stable else 'any'}",
                       f"network_{prefix}_{var}_{'stable' if stable else 'any'}_coverage"]])
    return pd.concat(rows,ignore_index=True)

network_exposure = pd.MultiIndex.from_product([YEARS,geo.nis],names=["year","nis"]).to_frame(index=False)
for var,direction in [
    ("tw_work_current","out"),
    ("bartik_work_2019","out"),
    ("tw_res_current","in"),
    ("bartik_res_2019","in"),
]:
    for stable in (False,True):
        z=network_weighted_exposure(exposure_panel,var,direction,stable)
        if len(z):
            network_exposure=merge_unique(network_exposure,z,label=f"network {var}")
write_table(network_exposure,SUPPORT/"network_weighted_telework_exposure.parquet")

# Spatial-weight matrices: exclude self, row-normalise; missing exposure values are renormalised at evaluation time.
N=len(id_order)
CONT=np.zeros((N,N),float)
idx={n:i for i,n in enumerate(id_order)}
for a,b in contig_edges:
    CONT[idx[a],idx[b]]=1; CONT[idx[b],idx[a]]=1
D_NOSELF=D.copy()
np.fill_diagonal(D_NOSELF,np.nan)
DIST50=np.exp(-np.nan_to_num(D_NOSELF,nan=np.inf)/50.0); np.fill_diagonal(DIST50,0)
INVD=np.divide(1.0,D_NOSELF,out=np.zeros_like(D_NOSELF),where=np.isfinite(D_NOSELF)&(D_NOSELF>0))
np.fill_diagonal(INVD,0)

def spatial_lag_for_var(panel,var,W,scheme):
    if var not in panel.columns:
        return pd.DataFrame()
    rows=[]
    for y in YEARS:
        x=panel[panel.year.eq(y)].set_index("nis")[var].reindex(id_order).to_numpy(float)
        valid=np.isfinite(x)
        denom=W[:,valid].sum(axis=1)
        val=np.divide(W[:,valid]@x[valid],denom,out=np.full(N,np.nan),where=denom>0)
        rows.append(pd.DataFrame({"year":y,"nis":id_order,f"spatial_{scheme}_{var}":val}))
    return pd.concat(rows,ignore_index=True)

spatial_exposure=pd.MultiIndex.from_product([YEARS,geo.nis],names=["year","nis"]).to_frame(index=False)
for var in ["tw_work_current","tw_res_current","bartik_work_2019","bartik_res_2019"]:
    for scheme,W in [("contig",CONT),("dist50",DIST50),("invdist",INVD)]:
        z=spatial_lag_for_var(exposure_panel,var,W,scheme)
        if len(z):
            spatial_exposure=merge_unique(spatial_exposure,z,label=f"spatial {var}")
write_table(spatial_exposure,SUPPORT/"spatial_telework_exposure.parquet")


## 14. Build the municipality × year master panel

In [ ]:

skeleton = pd.MultiIndex.from_product([YEARS,id_order],names=["year","nis"]).to_frame(index=False)
geo_attrs = geo.drop(columns="geometry").copy()
geo_attrs = geo_attrs[[c for c in [
    "nis","node_id","var_name","name_nl","name_fr","arr_nis","prov_nis","reg_nis",
    "boundary_reference","longitude","latitude","area_km2","ln_area_km2"
] if c in geo_attrs.columns]]
master = skeleton.merge(geo_attrs,on="nis",how="left",validate="many_to_one")

pop_keep = [c for c in population.columns if c in [
    "year","nis","population","population_male","population_female","population_density_km2",
    "ln_population","ln_population_density","population_reference_date"
]]
master=merge_unique(master,population[pop_keep],label="population")
master=merge_unique(master,socio,label="socio")
master=master.merge(transport_for_merge,on="nis",how="left",validate="many_to_one")
master=merge_unique(master,employment.drop(columns=["area_km2"],errors="ignore"),label="employment")
master=merge_unique(master,education_wide,label="education")
master=merge_unique(master,sex_wide,label="sex")
master=merge_unique(master,status_wide,label="status")
master=merge_unique(master,age_shares_wide,label="age")
master=merge_unique(master,heterogeneity_wide,label="extended heterogeneity")
master=master.merge(heterogeneity_baseline,on="nis",how="left",validate="many_to_one")
master=merge_unique(master,sector_summary_wide,label="sector")
master=merge_unique(master,resnet,label="residence network")
master=merge_unique(master,worknet,label="workplace network")
master=merge_unique(master,accessibility,label="accessibility")
master=merge_unique(master,exposure_panel,label="telework")
master=merge_unique(master,network_exposure,label="network exposure")
master=merge_unique(master,spatial_exposure,label="spatial exposure")

# Common baseline/pre-period versions. These are created for exploration and are explicitly tagged later.
BASELINE_CANDIDATES = [c for c in [
    "ln_population","ln_population_density","ln_housing_cost_main",
    "ln_taxable_income_per_resident","ln_net_income_per_resident","ln_adi_median_wapprox",
    "ln_employment_density_km2",
    "resident_education_high_share","workplace_education_high_share",
    "resident_sector_hhi_known","workplace_sector_hhi_known",
    "job_access_exp50","population_access_exp50",
] if c in master.columns]

for c in BASELINE_CANDIDATES:
    b = master.loc[master.year.eq(BASELINE_YEAR),["nis",c]].rename(columns={c:f"{c}_2019"})
    master=master.merge(b,on="nis",how="left",validate="many_to_one")
    pre = master.loc[master.year.isin(PRE_YEARS)].groupby("nis",as_index=False)[c].mean().rename(
        columns={c:f"{c}_premean_2015_2019"})
    master=master.merge(pre,on="nis",how="left",validate="many_to_one")

# One-year lags for a compact set of time-varying covariates.
LAG_VARS=[c for c in [
    "ln_population","ln_population_density","ln_housing_cost_main",
    "ln_taxable_income_per_resident","ln_net_income_per_resident","ln_adi_median_wapprox",
    "ln_employment_density_km2","job_access_exp50",
] if c in master.columns]
lag=master[["nis","year"]+LAG_VARS].copy()
lag["year"]+=1
lag=lag.rename(columns={c:"lag1_"+c for c in LAG_VARS})
master=master.merge(lag,on=["nis","year"],how="left",validate="one_to_one")

master["post_2020"]=master.year.ge(SHOCK_YEAR).astype("int8")
master["event_time_2020"]=master.year-SHOCK_YEAR
master=master.sort_values(["nis","year"]).reset_index(drop=True)

require(len(master)==EXPECTED_N*len(YEARS),"Municipality master is not 565 x 10")
require(master.groupby("year").nis.nunique().eq(EXPECTED_N).all(),"A year has fewer than 565 municipalities")

write_table(master,OUT/"01_municipality_year_master.parquet")
if WRITE_MUNICIPALITY_CSV:
    master.to_csv(OUT/"01_municipality_year_master.csv",index=False,encoding="utf-8-sig")

if WRITE_GEOJSON_2024:
    a=master[master.year.eq(2024)].copy()
    g=geo[["nis","geometry"]].merge(a,on="nis",how="left",validate="one_to_one")
    g=gpd.GeoDataFrame(g,geometry="geometry",crs=geo.crs)
    g.to_file(OUT/"municipality_master_2024.geojson",driver="GeoJSON")

print("Municipality master:",master.shape)


## 15. Build the OD pair × year master for network/PPML analysis

In [ ]:

OD_NODE_VARS=[c for c in [
    "population","ln_population","population_density_km2","ln_population_density",
    "housing_cost_main","ln_housing_cost_main","taxable_income_per_resident","ln_taxable_income_per_resident",
    "adi_median_wapprox","ln_adi_median_wapprox","employment_density_km2","ln_employment_density_km2",
    "resident_education_high_share","workplace_education_high_share",
    "tw_res_current","tw_work_current","bartik_res_2019","bartik_work_2019",
    "network_out_tw_work_current_stable","network_in_tw_res_current_stable",
    "spatial_contig_tw_res_current","spatial_contig_tw_work_current",
    "rail_stop_gravity_10km_2019","dist_nearest_rail_stop_km_2019",
    "mainline_rail_density_km_per_km2_2019","dist_nearest_motorway_link_km_2019",
    "major_road_density_km_per_km2_2019","rail_access_index_z_2019","road_access_index_z_2019",
] if c in master.columns]

HETERO_OD_VARS=[
    c for c in master.columns
    if c.endswith("_2019")
    and c.endswith("_share_2019")
    and any(c.startswith(p) for p in (
        "resident_nationality_","workplace_nationality_",
        "resident_origin_","workplace_origin_",
        "resident_generation_","resident_work_regime_",
        "resident_foreign_","workplace_foreign_"
    ))
]
OD_NODE_VARS=list(dict.fromkeys(OD_NODE_VARS+HETERO_OD_VARS))

o = master[["year","nis"]+OD_NODE_VARS].rename(columns={
    "nis":"home_nis", **{c:"o_"+c for c in OD_NODE_VARS}
})
d = master[["year","nis"]+OD_NODE_VARS].rename(columns={
    "nis":"work_nis", **{c:"d_"+c for c in OD_NODE_VARS}
})

od_master = od_domestic.merge(o,on=["year","home_nis"],how="left",validate="many_to_one")
od_master = od_master.merge(d,on=["year","work_nis"],how="left",validate="many_to_one")
od_master["post_2020"]=od_master.year.ge(SHOCK_YEAR).astype("int8")
od_master["event_time_2020"]=od_master.year-SHOCK_YEAR
od_master["ln_distance_plus1"]=np.log1p(od_master["distance_km"])

if {"o_tw_res_current","d_tw_work_current"}.issubset(od_master.columns):
    od_master["tw_pair_mean"]=(od_master["o_tw_res_current"]+od_master["d_tw_work_current"])/2
    od_master["tw_gap_work_minus_res"]=od_master["d_tw_work_current"]-od_master["o_tw_res_current"]
    od_master["distance_x_tw_work"]=od_master["distance_km"]*od_master["d_tw_work_current"]
    od_master["distance_x_tw_res"]=od_master["distance_km"]*od_master["o_tw_res_current"]
if {"o_bartik_res_2019","d_bartik_work_2019"}.issubset(od_master.columns):
    od_master["bartik_pair_mean"]=(od_master["o_bartik_res_2019"]+od_master["d_bartik_work_2019"])/2
    od_master["bartik_gap_work_minus_res"]=od_master["d_bartik_work_2019"]-od_master["o_bartik_res_2019"]

write_table(od_master,OUT/"02_od_pair_year_master.parquet")
if WRITE_OD_CSV_GZ:
    od_master.to_csv(OUT/"02_od_pair_year_master.csv.gz",index=False,encoding="utf-8-sig",compression="gzip")

if CREATE_BALANCED_PAIR_SKELETON:
    sk = pd.MultiIndex.from_product([YEARS,id_order,id_order],names=["year","home_nis","work_nis"]).to_frame(index=False)
    sk = sk.merge(pair,on=["home_nis","work_nis"],how="left",validate="many_to_one")
    obs = od_domestic[["year","home_nis","work_nis","workers","observed_in_source"]]
    sk = sk.merge(obs,on=["year","home_nis","work_nis"],how="left",validate="one_to_one")
    sk["observed_in_source"]=sk["observed_in_source"].fillna(False)
    sk["flow_missing_unpublished"]=sk["workers"].isna()
    if TREAT_UNPUBLISHED_OD_AS_ZERO:
        warnings.warn("You explicitly chose to treat unpublished OD cells as zero. This is an assumption, not a property verified from VAR.")
        sk["workers_assuming_unpublished_zero"]=sk["workers"].fillna(0)
    write_table(sk,SUPPORT/"od_balanced_pair_year_skeleton.parquet")
    print("Balanced pair skeleton:",sk.shape)

print("Observed OD master:",od_master.shape)


## 16. Variable roles, QA, source manifest and final exports

In [ ]:

def variable_role(c):
    lc=c.lower()
    if c in ("nis","year","node_id","var_name","home_nis","work_nis","pair_id"):
        return "identifier"
    hetero_tokens=("nationality_","origin_","generation_","work_regime_","foreign_nationality","foreign_origin")
    if any(t in lc for t in hetero_tokens) and lc.endswith("_2019"):
        return "baseline_heterogeneity"
    if any(t in lc for t in hetero_tokens) and lc.endswith("_share"):
        return "heterogeneity_candidate"
    if any(k in lc for k in ["rail_","railway_","road_","motorway","pt_stop"]) and lc.endswith("_2019"):
        return "baseline_transport_control"
    if lc.startswith("network_") and ("tw_" in lc or "bartik" in lc):
        return "network_exposure"
    if lc.startswith("spatial_") and ("tw_" in lc or "bartik" in lc):
        return "spatial_exposure"
    if "bartik" in lc or lc.startswith("tw_") or "telework" in lc:
        return "treatment_exposure"
    if "_2019" in lc or "_premean_2015_2019" in lc:
        return "baseline_control"
    if lc.startswith("lag1_"):
        return "lagged_control"
    if lc.startswith("resnet_") or lc.startswith("worknet_"):
        return "outcome_candidate"
    if "coverage" in lc or "gap_share" in lc or lc.endswith("_approximation"):
        return "qa"
    if any(k in lc for k in ["housing","income","population","employment_density","access_"]):
        return "time_varying_control_or_mechanism"
    return "descriptive_or_supporting"

def post_treatment_note(c,role):
    if role=="baseline_heterogeneity":
        return "Pre-treatment 2019 composition intended for interaction/heterogeneity specifications."
    if role=="heterogeneity_candidate":
        return "Annual composition descriptor; after 2020 it may be post-treatment. Prefer the corresponding 2019 baseline for causal heterogeneity."
    if role=="baseline_transport_control":
        return "Fixed 2019-01-01 OSM/Geofabrik infrastructure baseline; levels are absorbed by municipality FE. Use for matching/balancing, baseline × year interactions, heterogeneity, or robustness."
    if role=="time_varying_control_or_mechanism":
        return "May be affected by the 2020 shock; prefer pre-treatment baseline or lagged form in causal specifications unless a design justifies contemporaneous use."
    if role in ("network_exposure","spatial_exposure"):
        return "Exposure propagated through pre-specified weights where available; inspect definition before causal use."
    if role=="outcome_candidate":
        return "Network-derived outcome/descriptor; do not include contemporaneously as a control for the same network outcome."
    return ""

dictionary = pd.DataFrame({
    "variable":master.columns,
    "role":[variable_role(c) for c in master.columns],
})
dictionary["post_treatment_note"]=[post_treatment_note(c,r) for c,r in zip(dictionary.variable,dictionary.role)]
dictionary["source"]="master-derived or source-specific; see source manifest"
dictionary["definition"]=""
if "transport_codebook" in globals() and len(transport_codebook):
    for row in transport_codebook.to_dict("records"):
        mv=row.get("master_variable")
        if mv in set(dictionary.variable):
            dictionary.loc[dictionary.variable.eq(mv),"source"]="OpenStreetMap / Geofabrik Belgium historical snapshot 2019-01-01"
            dictionary.loc[dictionary.variable.eq(mv),"definition"]=str(row.get("definition",""))
dictionary.to_csv(OUT/"10_variable_dictionary.csv",index=False,encoding="utf-8-sig")

coverage=[]
for c in master.columns:
    if c in ("nis","year"):
        continue
    q=master.groupby("year")[c].apply(lambda s:s.notna().mean()).reset_index(name="share_nonmissing")
    q["variable"]=c
    coverage.append(q)
coverage=pd.concat(coverage,ignore_index=True)
coverage.to_csv(OUT/"11_data_coverage_QA.csv",index=False,encoding="utf-8-sig")

qa_rows=[
    {"check":"municipality_master_rows","value":len(master),"expected":EXPECTED_N*len(YEARS),"pass":len(master)==EXPECTED_N*len(YEARS)},
    {"check":"municipalities_each_year","value":int(master.groupby("year").nis.nunique().min()),"expected":EXPECTED_N,"pass":master.groupby("year").nis.nunique().eq(EXPECTED_N).all()},
    {"check":"pair_static_rows","value":len(pair),"expected":EXPECTED_N**2,"pass":len(pair)==EXPECTED_N**2},
    {"check":"transport_baseline_municipalities","value":transport_for_merge.nis.nunique(),"expected":EXPECTED_N,"pass":transport_for_merge.nis.nunique()==EXPECTED_N},
    {"check":"transport_main_candidate_missing_cells","value":int(transport_for_merge[[c+"_2019" for c in TRANSPORT_MAIN_CANDIDATES if c+"_2019" in transport_for_merge.columns]].isna().sum().sum()),"expected":0,"pass":int(transport_for_merge[[c+"_2019" for c in TRANSPORT_MAIN_CANDIDATES if c+"_2019" in transport_for_merge.columns]].isna().sum().sum())==0},
    {"check":"observed_od_negative_flows","value":int((od_master.workers<0).sum()),"expected":0,"pass":not (od_master.workers<0).any()},
]

if BUILD_EXTENDED_HETEROGENEITY:
    hetero_vals=heterogeneity_wide[
        [c for c in heterogeneity_wide.columns if c.endswith("_share")]
    ].apply(pd.to_numeric,errors="coerce").stack()
    qa_rows.extend([
        {
            "check":"extended_heterogeneity_nis_outside_canonical",
            "value":int((~heterogeneity_long["nis"].isin(set(geo["nis"]))).sum()) if len(heterogeneity_long) else 0,
            "expected":0,
            "pass":bool((heterogeneity_long.empty) or heterogeneity_long["nis"].isin(set(geo["nis"])).all()),
        },
        {
            "check":"extended_heterogeneity_share_outside_0_1",
            "value":int(((hetero_vals<0)|(hetero_vals>1)).sum()) if len(hetero_vals) else 0,
            "expected":0,
            "pass":bool((not len(hetero_vals)) or (((hetero_vals>=0)&(hetero_vals<=1)).all())),
        },
        {
            "check":"extended_heterogeneity_families_complete",
            "value":int(heterogeneity_family_summary["extraction_ok"].sum()),
            "expected":len(EXPECTED_HETERO_FAMILIES),
            "pass":bool(heterogeneity_family_summary["extraction_ok"].all()),
        },
    ])

if BUILD_FULL_VAR_CATALOG and not var_catalog.empty:
    final_finished = var_catalog.status.isin(["received","empty_response_not_zero"])
    manifest_task_count = int(var_plan_manifest.get("task_count", len(var_catalog)))
    qa_rows.extend([
        {
            "check":"var_final_plan_task_count",
            "value":len(var_catalog),
            "expected":manifest_task_count,
            "pass":len(var_catalog)==manifest_task_count,
        },
        {
            "check":"var_final_plan_unfinished_tasks",
            "value":int((~final_finished).sum()),
            "expected":0,
            "pass":bool(final_finished.all()),
        },
        {
            "check":"var_collector_summary_complete",
            "value":bool(var_collection_summary.get("all_planned_queries_received",False)),
            "expected":True,
            "pass":bool(var_collection_summary.get("all_planned_queries_received",False)),
        },
    ])

if "exposure_coverage_audit" in globals() and len(exposure_coverage_audit):
    audited_missing = int(exposure_coverage_audit["n_missing"].sum())
    documented_missing = len(exposure_missing_detail)
    qa_rows.append({
        "check":"telework_exposure_missing_cells_documented",
        "value":documented_missing,
        "expected":audited_missing,
        "pass":documented_missing==audited_missing,
    })
if not exposure_panel.empty:
    rate_like = [
        c for c in exposure_panel.columns
        if c.startswith("tw_")
        and "workers" not in c
        and "coverage" not in c
        and (
            "current" in c
            or "frozen" in c
            or re.search(r"_s20\d{2}_r20\d{2}$", c)
        )
    ]
    if rate_like:
        bad=0
        for c in rate_like:
            s=pd.to_numeric(exposure_panel[c],errors="coerce").dropna()
            bad += int(((s<0)|(s>1)).sum())
        qa_rows.append({"check":"telework_rate_like_exposure_outside_0_1","value":bad,"expected":0,"pass":bad==0})
qa=pd.DataFrame(qa_rows)
qa.to_csv(QA_DIR/"master_integrity_checks.csv",index=False,encoding="utf-8-sig")

sources = {
    "basemap":BASEMAP,"population_raw":POP_XLSX,"population_existing":POP_PANEL_EXISTING,
    "housing":HOUSE_XLSX,"tax":TAX_XLSX,"adi":ADI_XLSX,
    **{"od_"+k:v for k,v in OD_FILES.items()},
    "var_direct_root":VAR_DIRECT_ROOT,"telework_original":TELEWORK_XLSX,
    "wfh_rate_component":WFH_RATE_CSV,"wfh_workplace_margin_component":WFH_WORKPLACE_MARGIN_CSV,
    "transport_controls_2019":TRANSPORT_CSV_EXISTING,
    "transport_codebook_2019":TRANSPORT_CODEBOOK_EXISTING,
    "transport_metadata_2019":TRANSPORT_METADATA_EXISTING,
    "transport_osm_geofabrik_2019_archive":TRANSPORT_ZIP,
    "var_final_plan":VAR_FINAL_PLAN,
    "var_plan_manifest":VAR_PLAN_MANIFEST,
    "var_task_status":VAR_TASK_STATUS,
    "var_collection_summary":VAR_COLLECTION_SUMMARY,
}
manifest=[]
for name,p in sources.items():
    p=Path(p)
    manifest.append({
        "source":name,"path":str(p),"exists":p.exists(),
        "size_mb":round(p.stat().st_size/1024**2,3) if p.is_file() else np.nan,
        "note":"directory" if p.is_dir() else ""
    })
pd.DataFrame(manifest).to_csv(OUT/"12_source_manifest.csv",index=False,encoding="utf-8-sig")

readme = f"""Belgium master data preparation v1.3.1
Generated from: {ROOT}
Years: {YEAR_MIN}-{YEAR_MAX}
Canonical geography: {EXPECTED_N} municipalities, boundary reference 2025-01-01

Main outputs
01_municipality_year_master.parquet
02_od_pair_year_master.parquet

Supporting outputs include pair-static distance/contiguity, fixed 2019 rail/road transport accessibility, residence/workplace network metrics, extended nationality/origin/generation/work-regime composition,
telework exposures, pre-2019 commuting weights, spatial weights, sector panels, VAR direct archive inventory,
coverage QA and variable roles.

Important:
- Missing/unpublished OD cells are NOT changed to zero by default.
- Municipality self-links have representative-point distance 0; this is not an estimate of within-municipality travel distance.
- Rail/road transport controls are fixed at the 2019-01-01 OSM/Geofabrik baseline and are pre-treatment characteristics.\n- Current housing, income, employment accessibility and network characteristics may be post-treatment variables after 2020.
- Baseline 2019 and 2015-2019 pre-period means are supplied for causal designs.
- Final VAR collection-plan tasks are separated from historical probe/cache requests.\n- Full VAR marginal tables are indexed but are not cross-joined into fictitious joint distributions.\n- Missing telework exposure cells are not imputed; municipality-level causes are written to QA diagnostics.\n- Nationality, origin, generation and work-regime margins are kept separate; no fictitious joint demographic distribution is constructed.\n- For causal heterogeneity, prefer 2019 composition shares over contemporaneous post-2020 shares.
"""
(OUT/"README_master_data.txt").write_text(readme,encoding="utf-8")

display(qa)
print("Saved:")
for p in [
    OUT/"01_municipality_year_master.parquet",
    OUT/"02_od_pair_year_master.parquet",
    OUT/"10_variable_dictionary.csv",
    OUT/"11_data_coverage_QA.csv",
    OUT/"12_source_manifest.csv",
]:
    print(" ",p)


## 17. Optional quick exploration checks

In [ ]:

# These plots are descriptive QA only.
plt.rcParams.update({"font.family":"serif","font.serif":["Times New Roman","DejaVu Serif"],
                     "axes.spines.top":False,"axes.spines.right":False})

fig,ax=plt.subplots(figsize=(7.2,4.2))
annual=od_domestic.groupby("year")["workers"].sum()
ax.plot(annual.index,annual.values,marker="o")
ax.set(xlabel="Year",ylabel="Published domestic worker flow",title="Belgian job–home network coverage over time")
fig.tight_layout()
plt.show()

expcols=[c for c in ["tw_res_current","tw_work_current","bartik_res_2019","bartik_work_2019"] if c in master.columns]
if expcols:
    display(master.groupby("year")[expcols].agg(["mean","std","count"]).round(4))

display(master[["year","nis"]+[c for c in [
    "resnet_mean_km","resnet_external_mean_km","resnet_cross_share",
    "worknet_mean_km","worknet_external_mean_km","worknet_cross_share"
] if c in master.columns]].head())


transport_show=[c for c in ["rail_stop_gravity_10km_2019","dist_nearest_rail_stop_km_2019","dist_nearest_motorway_link_km_2019","major_road_density_km_per_km2_2019","rail_access_index_z_2019","road_access_index_z_2019"] if c in master.columns]
if transport_show:
    display(master.loc[master.year.eq(2019),["nis"]+transport_show].describe(include="all").T)


if BUILD_FULL_VAR_CATALOG and not var_catalog.empty:
    print("\nFinal VAR collection-plan status counts:")
    display(
        var_catalog["status"].value_counts(dropna=False)
        .rename_axis("status").reset_index(name="tasks")
    )
    print("Historical cache metadata rows (provenance only):", len(var_request_cache))

if "exposure_coverage_audit" in globals() and len(exposure_coverage_audit):
    print("\nLowest telework-exposure coverage:")
    display(
        exposure_coverage_audit.sort_values(
            ["share_nonmissing","variable","year"]
        ).head(20)
    )


if BUILD_EXTENDED_HETEROGENEITY and "heterogeneity_coverage" in globals() and len(heterogeneity_coverage):
    print("\nExtended heterogeneity coverage:")
    display(
        heterogeneity_coverage.sort_values(["side","dimension","year"])
    )
    baseline_preview=[
        c for c in master.columns
        if c.endswith("_share_2019")
        and any(t in c for t in [
            "nationality_","origin_","generation_","work_regime_","foreign_"
        ])
    ]
    if baseline_preview:
        print("\n2019 heterogeneity baseline variables:",len(baseline_preview))
        display(master.loc[master.year.eq(2019),["nis"]+baseline_preview[:12]].head())
